# Figure 1

Simulated annealing (SA) and parallel tempering (PT) runs for 32-site (8 linker types) and 72-site (4 linker types).



In [1]:
# Energy Trajectory Comparison: SA vs Parallel Tempering

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import time
from itertools import permutations
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
SEED = 42
np.random.seed(SEED)
STEPS = 10000  # For SA trajectory matching paper

# PT Parameters
NUM_REPLICAS = 16
T_MIN = 0.1
T_MAX = 100.0
EXCHANGE_INTERVAL = 100
PT_STEPS = 10000  # Same as SA for fair comparison

# Material Parameters (from Kang et al.)
MATERIAL_PARAMS = {
    'topological': {'alpha': 1.0, 'distance': 3.0},
    'spatial': {'alpha': 0.01, 'distance': 5.2}
}

# Linker Definitions
TYPES_16 = ['THQ', 'HHTP']
COUNTS_16 = {'THQ': 8, 'HHTP': 8}
LENGTHS_16 = {'THQ': 2.42, 'HHTP': 4.87}

TYPES_32 = ['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8']
COUNTS_32 = {t: 4 for t in TYPES_32}
LENGTHS_32 = {
    'L1': 2.0, 'L2': 3.0, 'L3': 4.0, 'L4': 5.0,
    'L5': 6.0, 'L6': 7.0, 'L7': 8.0, 'L8': 9.0
}

TYPES_72 = ['THQ', 'HHTP', 'HHTT', 'HHTN']
COUNTS_72 = {t: 18 for t in TYPES_72}
LENGTHS_72 = {'THQ': 2.0, 'HHTP': 4.0, 'HHTT': 6.0, 'HHTN': 8.0}

# ============================================================================
# GRAPH UTILITIES
# ============================================================================
def add_edge_weights_to_graph(G, params=MATERIAL_PARAMS):
    """CORRECTED: Distance-based spatial edge detection"""
    t_p = params['topological']
    s_p = params['spatial']
    w_topo = t_p['distance'] ** t_p['alpha']
    w_spatial = s_p['distance'] ** s_p['alpha']

    # Get positions
    pos = nx.get_node_attributes(G, 'pos')
    if not pos:
        pos = {node: data['pos'] for node, data in G.nodes(data=True) if 'pos' in data}

    # Mark topological edges
    topo_distances = []
    for u, v in G.edges():
        G[u][v]['weight'] = w_topo
        G[u][v]['edge_type'] = 'topological'
        pos_u = np.array(pos[u])
        pos_v = np.array(pos[v])
        dist = np.linalg.norm(pos_u - pos_v)
        topo_distances.append(dist)

    # Find spatial edges by distance (2nd-nearest neighbors)
    avg_topo_dist = np.mean(topo_distances)
    spatial_distance_min = 1.4 * avg_topo_dist
    spatial_distance_max = 2.1 * avg_topo_dist

    nodes = list(G.nodes())
    spatial_edges = set()

    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            if G.has_edge(u, v):
                continue
            pos_u = np.array(pos[u])
            pos_v = np.array(pos[v])
            dist = np.linalg.norm(pos_u - pos_v)
            if spatial_distance_min <= dist <= spatial_distance_max:
                spatial_edges.add((u, v))

    for u, v in spatial_edges:
        G.add_edge(u, v, weight=w_spatial, edge_type='spatial')

    return G

class MTVSolver:
    def __init__(self, graph, target_counts, linker_lengths):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        self.graph = nx.relabel_nodes(graph, mapping)
        self.edges = np.array(list(self.graph.edges()), dtype=int)
        self.linkers = list(target_counts.keys())
        self.lengths = linker_lengths
        self.length_lookup = np.array([linker_lengths[l] for l in self.linkers])
        self.edge_weights = np.array([self.graph[i][j].get('weight', 1.0) for i, j in self.edges])

        self.state_pool = []
        for linker_idx, linker_name in enumerate(self.linkers):
            self.state_pool.extend([linker_idx] * target_counts[linker_name])
        self.state_pool = np.array(self.state_pool)

    def get_energy(self, current_state):
        u = self.edges[:, 0]
        v = self.edges[:, 1]
        len_u = self.length_lookup[current_state[u]]
        len_v = self.length_lookup[current_state[v]]
        edge_lengths = len_u + len_v
        mean_length = np.mean(edge_lengths)
        return np.sum(self.edge_weights * (edge_lengths - mean_length) ** 2)

    def solve_with_history(self, steps=STEPS, initial_temp=100.0):
        """Standard SA with energy history tracking"""
        current_state = self.state_pool.copy()
        np.random.shuffle(current_state)
        current_energy = self.get_energy(current_state)
        best_state = current_state.copy()
        best_energy = current_energy
        history = []
        n_sites = len(current_state)

        for i in range(steps):
            history.append(best_energy)

            idx1, idx2 = np.random.randint(0, n_sites, 2)
            if current_state[idx1] == current_state[idx2]:
                continue

            current_state[idx1], current_state[idx2] = current_state[idx2], current_state[idx1]
            new_energy = self.get_energy(current_state)
            delta = new_energy - current_energy
            temp = initial_temp * (0.95 ** (i / 100.0))

            if delta < 0 or np.random.rand() < np.exp(-delta / (temp + 1e-10)):
                current_energy = new_energy
                if current_energy < best_energy:
                    best_energy = current_energy
                    best_state = current_state.copy()
            else:
                current_state[idx1], current_state[idx2] = current_state[idx2], current_state[idx1]

        return history, best_state

class ParallelTemperingSolver:
    def __init__(self, graph, target_counts, linker_lengths):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        self.graph = nx.relabel_nodes(graph, mapping)

        self.edges = np.array(list(self.graph.edges()), dtype=int)
        self.linkers = list(target_counts.keys())
        self.lengths = linker_lengths
        self.length_lookup = np.array([linker_lengths[l] for l in self.linkers])
        self.edge_weights = np.array([self.graph[i][j].get('weight', 1.0) for i, j in self.edges])

        self.state_pool = []
        for linker_idx, linker_name in enumerate(self.linkers):
            self.state_pool.extend([linker_idx] * target_counts[linker_name])
        self.state_pool = np.array(self.state_pool)
        self.n_sites = len(self.state_pool)

        # Temperature ladder
        self.temperatures = np.logspace(np.log10(T_MIN), np.log10(T_MAX), NUM_REPLICAS)
        self.betas = 1.0 / self.temperatures

    def get_energy(self, current_state):
        u = self.edges[:, 0]
        v = self.edges[:, 1]
        len_u = self.length_lookup[current_state[u]]
        len_v = self.length_lookup[current_state[v]]
        edge_lengths = len_u + len_v
        mean_length = np.mean(edge_lengths)
        return np.sum(self.edge_weights * (edge_lengths - mean_length) ** 2)

    def solve_with_history(self, steps=PT_STEPS):
        """PT with tracking of best energy across all replicas"""
        # Initialize replicas
        replicas = []
        energies = []

        for _ in range(NUM_REPLICAS):
            state = self.state_pool.copy()
            np.random.shuffle(state)
            replicas.append(state.copy())
            energies.append(self.get_energy(state))

        # Track global best (monotonically decreasing)
        best_energy_history = []
        global_best = min(energies)
        best_state_global = replicas[energies.index(global_best)].copy()

        # Main loop
        for step in range(steps):
            # Update global best if we found something better
            current_best = min(energies)
            if current_best < global_best:
                global_best = current_best
                best_state_global = replicas[energies.index(current_best)].copy()
            best_energy_history.append(global_best)

            # MC moves for each replica
            for i in range(NUM_REPLICAS):
                idx1, idx2 = np.random.randint(0, self.n_sites, 2)
                if replicas[i][idx1] == replicas[i][idx2]:
                    continue

                current_E = energies[i]
                replicas[i][idx1], replicas[i][idx2] = replicas[i][idx2], replicas[i][idx1]
                new_E = self.get_energy(replicas[i])
                delta_E = new_E - current_E
                beta = self.betas[i]

                if delta_E < 0 or np.random.rand() < np.exp(-beta * delta_E):
                    energies[i] = new_E
                else:
                    replicas[i][idx1], replicas[i][idx2] = replicas[i][idx2], replicas[i][idx1]

            # Replica exchanges
            if step % EXCHANGE_INTERVAL == 0 and step > 0:
                offset = (step // EXCHANGE_INTERVAL) % 2
                for i in range(offset, NUM_REPLICAS - 1, 2):
                    j = i + 1
                    beta_i = self.betas[i]
                    beta_j = self.betas[j]
                    E_i = energies[i]
                    E_j = energies[j]

                    delta_beta = beta_i - beta_j
                    delta_E = E_j - E_i
                    log_acceptance = delta_beta * delta_E

                    if log_acceptance >= 0 or np.random.rand() < np.exp(log_acceptance):
                        replicas[i], replicas[j] = replicas[j], replicas[i]
                        energies[i], energies[j] = energies[j], energies[i]

        return best_energy_history, best_state_global

def visualize_configuration_with_similarity(state, other_state, graph, linkers, lengths,
                                           title="Configuration", ax=None, show_energy=None,
                                           highlight_same=False, show_legend=True):
    """Visualize configuration with similarity highlighting (green border for same assignments)"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 7))

    # Ensure integer node labels
    if not all(isinstance(node, int) for node in graph.nodes()):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        graph = nx.relabel_nodes(graph, mapping)

    pos = nx.get_node_attributes(graph, 'pos')
    if not pos:
        pos = nx.spring_layout(graph, seed=SEED)

    # Normalize positions
    pos_array = np.array(list(pos.values()))
    pos_min = pos_array.min(axis=0)
    pos_max = pos_array.max(axis=0)
    pos_range = pos_max - pos_min
    pos_normalized = {node: (p - pos_min) / pos_range for node, p in pos.items()}

    # Draw edges
    for i, j, data in graph.edges(data=True):
        edge_type = data.get('edge_type', 'unknown')
        x_coords = [pos_normalized[i][0], pos_normalized[j][0]]
        y_coords = [pos_normalized[i][1], pos_normalized[j][1]]

        if edge_type == 'topological':
            ax.plot(x_coords, y_coords, color='#333333', linewidth=2.5, alpha=0.9, zorder=5)
        else:
            ax.plot(x_coords, y_coords, color='#CCCCCC', linewidth=1.0, alpha=0.5, zorder=1)

    # Determine if we have 4 or 8 linker types
    unique_linkers = sorted(set(linkers))

    if len(unique_linkers) == 2:
        # 2 types (16-site system)
        colors = {
            'THQ': '#2ca02c', 'HHTP': '#d62728',
            'T': '#2ca02c', 'H': '#d62728'
        }
        node_size = 0.050  # Increased from 0.040
        font_size = 12  # Increased from 10
        show_labels = True
    elif len(unique_linkers) == 4:
        # 4 types with specific colors matching your figure
        colors = {
            # For 72-site (4 types)
            'THQ': '#2ca02c', 'HHTP': '#d62728', 'HHTT': '#1f77b4', 'HHTN': '#ff7f0e',
            'T': '#2ca02c', 'H': '#d62728'
        }
        node_size = 0.040
        font_size = 10
        show_labels = True
    elif len(unique_linkers) == 8:
        # 8 types (32-site) with specific colors and letter labels
        colors = {
            'L1': '#2ca02c', 'L2': '#d62728', 'L3': '#1f77b4', 'L4': '#ff7f0e',
            'L5': '#9467bd', 'L6': '#8c564b', 'L7': '#e377c2', 'L8': '#7f7f7f'
        }
        # Map to single letters for display
        letter_map = {
            'L1': '1', 'L2': '2', 'L3': '3', 'L4': '4',
            'L5': '5', 'L6': '6', 'L7': '7', 'L8': '8'
        }
        node_size = 0.040  # Increased from 0.032
        font_size = 11  # Increased from 9
        show_labels = True
    else:
        # Fallback
        colors_map = plt.cm.tab10(np.linspace(0, 1, len(unique_linkers)))
        colors = {linker: colors_map[i] for i, linker in enumerate(unique_linkers)}
        node_size = 0.025
        font_size = 5
        show_labels = False

    # Draw nodes with similarity highlighting
    for node_idx in graph.nodes():
        x, y = pos_normalized[node_idx]
        linker = linkers[state[node_idx]]
        color = colors.get(linker, 'gray')

        # Check if this node has the same assignment in both states
        is_same = (state[node_idx] == other_state[node_idx]) if highlight_same else False

        # Border color: lime green if same, black if different
        border_color = 'lime' if is_same else 'black'
        border_width = 3 if is_same else 1.5

        circle = plt.Circle((x, y), node_size, color=color, alpha=0.95,
                          ec=border_color, linewidth=border_width, zorder=10)
        ax.add_patch(circle)

        if show_labels:
            # Show first letter for 2-type and 4-type systems
            if len(unique_linkers) in [2, 4]:
                label_text = linker[0] if len(linker) > 0 else linker
            # Show numbers for 8-type system
            elif len(unique_linkers) == 8:
                label_text = letter_map.get(linker, linker[-1])
            else:
                label_text = linker[0] if len(linker) > 0 else linker

            ax.text(x, y, label_text,
                   ha='center', va='center', fontsize=font_size,
                   fontweight='bold', color='white', zorder=11)

    ax.set_aspect('equal')
    ax.axis('off')

    title_text = f'{title}'
    if show_energy is not None:
        title_text += f'\nEnergy = {show_energy:.2f}'
    ax.set_title(title_text, fontsize=11, fontweight='bold', pad=10)

    # Add legend for linker types and edge types (only if show_legend=True)
    if show_legend:
        legend_elements = []

        # Add linker type legend items
        if len(unique_linkers) in [2, 4]:
            # Show actual linker names with their lengths
            for linker in unique_linkers:
                length = lengths.get(linker, 0)
                legend_elements.append(Line2D([0], [0], marker='o', color='w',
                                             markerfacecolor=colors[linker],
                                             markersize=10,
                                             label=f'{linker} ({length:.2f}Å)',
                                             markeredgecolor='black', markeredgewidth=1))
        elif len(unique_linkers) == 8:
            # Show linker names with numbers and lengths for 8-type system
            for linker in unique_linkers:
                length = lengths.get(linker, 0)
                legend_elements.append(Line2D([0], [0], marker='o', color='w',
                                             markerfacecolor=colors[linker],
                                             markersize=10,
                                             label=f'{linker} ({length:.1f}Å)',
                                             markeredgecolor='black', markeredgewidth=1))

        # Add edge type legend items
        legend_elements.append(Line2D([0], [0], color='#333333', linewidth=2.5,
                                     label='Topological', alpha=0.9))
        legend_elements.append(Line2D([0], [0], color='#CCCCCC', linewidth=1.0,
                                     label='Spatial', alpha=0.5))

        # Place legend at bottom center
        if len(unique_linkers) == 8:
            # More items for 8-type, use 5 columns
            ax.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.25),
                     framealpha=0.9, fontsize=10, ncol=5)
        else:
            ax.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.20),
                     framealpha=0.9, fontsize=11, ncol=3)

def plot_comparison_figure():
    np.random.seed(SEED)

    print("="*70)
    print("Energy Trajectory Comparison: SA vs Parallel Tempering")
    print("="*70)

    # 16-Site System
    print(f"\n[16-Site System - 4 Linker Types]")
    G16 = add_edge_weights_to_graph(nx.hexagonal_lattice_graph(2, 4, periodic=True))
    print(f"  Graph: {len(G16.nodes())} nodes, {len(G16.edges())} edges")

    print("  Running SA...")
    solver16_sa = MTVSolver(G16, COUNTS_16, LENGTHS_16)
    t0 = time.time()
    hist16_sa, state16_sa = solver16_sa.solve_with_history(steps=STEPS)
    t1 = time.time()
    print(f"    Time: {t1-t0:.2f}s | Final Energy: {hist16_sa[-1]:.4f}")

    print("  Running PT...")
    solver16_pt = ParallelTemperingSolver(G16, COUNTS_16, LENGTHS_16)
    t0 = time.time()
    hist16_pt, state16_pt = solver16_pt.solve_with_history(steps=PT_STEPS)
    t1 = time.time()
    print(f"    Time: {t1-t0:.2f}s | Final Energy: {hist16_pt[-1]:.4f}")

    # 32-Site System
    print(f"\n[32-Site System - 8 Linker Types]")
    G32 = add_edge_weights_to_graph(nx.hexagonal_lattice_graph(4, 4, periodic=True))
    print(f"  Graph: {len(G32.nodes())} nodes, {len(G32.edges())} edges")

    print("  Running SA...")
    solver32_sa = MTVSolver(G32, COUNTS_32, LENGTHS_32)
    t0 = time.time()
    hist32_sa, state32_sa = solver32_sa.solve_with_history(steps=STEPS)
    t1 = time.time()
    print(f"    Time: {t1-t0:.2f}s | Final Energy: {hist32_sa[-1]:.4f}")

    print("  Running PT...")
    solver32_pt = ParallelTemperingSolver(G32, COUNTS_32, LENGTHS_32)
    t0 = time.time()
    hist32_pt, state32_pt = solver32_pt.solve_with_history(steps=PT_STEPS)
    t1 = time.time()
    print(f"    Time: {t1-t0:.2f}s | Final Energy: {hist32_pt[-1]:.4f}")


    # 72-Site System
    print(f"\n[72-Site System - 4 Linker Types]")
    G72 = add_edge_weights_to_graph(nx.hexagonal_lattice_graph(6, 6, periodic=True))
    print(f"  Graph: {len(G72.nodes())} nodes, {len(G72.edges())} edges")

    print("  Running SA...")
    solver72_sa = MTVSolver(G72, COUNTS_72, LENGTHS_72)
    t0 = time.time()
    hist72_sa, state72_sa = solver72_sa.solve_with_history(steps=STEPS)
    t1 = time.time()
    print(f"    Time: {t1-t0:.2f}s | Final Energy: {hist72_sa[-1]:.4f}")

    print("  Running PT...")
    solver72_pt = ParallelTemperingSolver(G72, COUNTS_72, LENGTHS_72)
    t0 = time.time()
    hist72_pt, state72_pt = solver72_pt.solve_with_history(steps=PT_STEPS)
    t1 = time.time()
    print(f"    Time: {t1-t0:.2f}s | Final Energy: {hist72_pt[-1]:.4f}")

    # ======================================================================
    # FIGURE 1: ENERGY TRAJECTORIES (All three systems)
    # MODIFIED: Consistent colors - Blue for SA, Orange for PT
    # ======================================================================
    print("\n[Generating energy trajectory comparison...]")

    fig1, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 14))

    # 16-Site Trajectory - CONSISTENT COLORS
    ax1.plot(hist16_sa, label='SA', linewidth=2.5, color='#1f77b4', alpha=0.9)
    ax1.plot(hist16_pt, label='Parallel Tempering', linewidth=2.5, color='#ff7f0e', alpha=0.9)
    ax1.axhline(y=hist16_sa[-1], color='#1f77b4', linestyle='--', alpha=0.5, linewidth=1.5)
    ax1.axhline(y=hist16_pt[-1], color='#ff7f0e', linestyle='--', alpha=0.5, linewidth=1.5)
    ax1.set_xlabel('Iteration Step', fontsize=13)
    ax1.set_ylabel('Best Energy Found', fontsize=13)
    ax1.set_title(f'16 Sites (2 Types) \nSA: {hist16_sa[-1]:.2f} | PT: {hist16_pt[-1]:.2f}',
                 fontsize=14, fontweight='bold', pad=15)
    ax1.legend(fontsize=12, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, STEPS)

    # 32-Site Trajectory - CONSISTENT COLORS
    ax2.plot(hist32_sa, label='SA', linewidth=2.5, color='#1f77b4', alpha=0.9)
    ax2.plot(hist32_pt, label='Parallel Tempering', linewidth=2.5, color='#ff7f0e', alpha=0.9)
    ax2.axhline(y=hist32_sa[-1], color='#1f77b4', linestyle='--', alpha=0.5, linewidth=1.5)
    ax2.axhline(y=hist32_pt[-1], color='#ff7f0e', linestyle='--', alpha=0.5, linewidth=1.5)
    ax2.set_xlabel('Iteration Step', fontsize=13)
    ax2.set_ylabel('Best Energy Found', fontsize=13)
    ax2.set_title(f'32 Sites (8 Types) \nSA: {hist32_sa[-1]:.2f} | PT: {hist32_pt[-1]:.2f}',
                 fontsize=14, fontweight='bold', pad=15)
    ax2.legend(fontsize=12, loc='upper right')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, STEPS)

    # 72-Site Trajectory - CONSISTENT COLORS
    ax3.plot(hist72_sa, label='SA', linewidth=2.5, color='#1f77b4', alpha=0.9)
    ax3.plot(hist72_pt, label='Parallel Tempering', linewidth=2.5, color='#ff7f0e', alpha=0.9)
    ax3.axhline(y=hist72_sa[-1], color='#1f77b4', linestyle='--', alpha=0.5, linewidth=1.5)
    ax3.axhline(y=hist72_pt[-1], color='#ff7f0e', linestyle='--', alpha=0.5, linewidth=1.5)
    ax3.set_xlabel('Iteration Step', fontsize=13)
    ax3.set_ylabel('Best Energy Found', fontsize=13)
    ax3.set_title(f'72 Sites (4 Types) \nSA: {hist72_sa[-1]:.2f} | PT: {hist72_pt[-1]:.2f}',
                 fontsize=14, fontweight='bold', pad=15)
    ax3.legend(fontsize=12, loc='upper right')
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim(0, STEPS)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    output1 = 'Energy_Trajectories_SA_vs_PT.png'
    plt.savefig(output1, dpi=300, bbox_inches='tight')
    print(f"  Saved: {output1}")
    plt.close(fig1)

    # ======================================================================
    # FIGURE 2: CONFIGURATIONS (3 rows x 2 columns: SA left, PT right)
    # ======================================================================
    print("\n[Generating configuration comparison...]")

    fig2 = plt.figure(figsize=(16, 20))
    gs2 = fig2.add_gridspec(3, 2, hspace=0.35, wspace=0.12)

    # Row 1: 16-Site
    ax_16sa = fig2.add_subplot(gs2[0, 0])
    ax_16pt = fig2.add_subplot(gs2[0, 1])

    # Row 2: 32-Site
    ax_32sa = fig2.add_subplot(gs2[1, 0])
    ax_32pt = fig2.add_subplot(gs2[1, 1])

    # Row 3: 72-Site
    ax_72sa = fig2.add_subplot(gs2[2, 0])
    ax_72pt = fig2.add_subplot(gs2[2, 1])

    # 16-Site Configurations (no legend on individual plots)
    visualize_configuration_with_similarity(state16_sa, state16_pt, solver16_sa.graph,
                                           TYPES_16, LENGTHS_16,
                                           title='16-Site SA',
                                           ax=ax_16sa, show_energy=hist16_sa[-1],
                                           show_legend=False)

    visualize_configuration_with_similarity(state16_pt, state16_sa, solver16_pt.graph,
                                           TYPES_16, LENGTHS_16,
                                           title='16-Site PT',
                                           ax=ax_16pt, show_energy=hist16_pt[-1],
                                           show_legend=False)

    # 32-Site Configurations (no legend on individual plots)
    visualize_configuration_with_similarity(state32_sa, state32_pt, solver32_sa.graph,
                                           TYPES_32, LENGTHS_32,
                                           title='32-Site SA',
                                           ax=ax_32sa, show_energy=hist32_sa[-1],
                                           show_legend=False)

    visualize_configuration_with_similarity(state32_pt, state32_sa, solver32_pt.graph,
                                           TYPES_32, LENGTHS_32,
                                           title='32-Site PT',
                                           ax=ax_32pt, show_energy=hist32_pt[-1],
                                           show_legend=False)

    # 72-Site Configurations (no legend on individual plots)
    visualize_configuration_with_similarity(state72_sa, state72_pt, solver72_sa.graph,
                                           TYPES_72, LENGTHS_72,
                                           title='72-Site SA',
                                           ax=ax_72sa, show_energy=hist72_sa[-1],
                                           show_legend=False)

    visualize_configuration_with_similarity(state72_pt, state72_sa, solver72_pt.graph,
                                           TYPES_72, LENGTHS_72,
                                           title='72-Site PT',
                                           ax=ax_72pt, show_energy=hist72_pt[-1],
                                           show_legend=False)

    # Add centered legends for each row
    # Legend for 16-site (Row 1)
    legend_elements_16 = []
    for linker in TYPES_16:
        length = LENGTHS_16.get(linker, 0)
        color = '#2ca02c' if linker == 'THQ' else '#d62728'
        legend_elements_16.append(Line2D([0], [0], marker='o', color='w',
                                     markerfacecolor=color,
                                     markersize=12,
                                     label=f'{linker} ({length:.2f}Å)',
                                     markeredgecolor='black', markeredgewidth=1))
    legend_elements_16.append(Line2D([0], [0], color='#333333', linewidth=2.5,
                                 label='Topological', alpha=0.9))
    legend_elements_16.append(Line2D([0], [0], color='#CCCCCC', linewidth=1.0,
                                 label='Spatial', alpha=0.5))

    legend_16 = fig2.legend(handles=legend_elements_16, loc='center',
                           bbox_to_anchor=(0.5, 0.64), framealpha=0.9, fontsize=11, ncol=4)

    # Legend for 32-site (Row 2)
    legend_elements_32 = []
    colors_32 = {
        'L1': '#2ca02c', 'L2': '#d62728', 'L3': '#1f77b4', 'L4': '#ff7f0e',
        'L5': '#9467bd', 'L6': '#8c564b', 'L7': '#e377c2', 'L8': '#7f7f7f'
    }
    for linker in TYPES_32:
        length = LENGTHS_32.get(linker, 0)
        legend_elements_32.append(Line2D([0], [0], marker='o', color='w',
                                     markerfacecolor=colors_32[linker],
                                     markersize=12,
                                     label=f'{linker} ({length:.1f}Å)',
                                     markeredgecolor='black', markeredgewidth=1))
    legend_elements_32.append(Line2D([0], [0], color='#333333', linewidth=2.5,
                                 label='Topological', alpha=0.9))
    legend_elements_32.append(Line2D([0], [0], color='#CCCCCC', linewidth=1.0,
                                 label='Spatial', alpha=0.5))

    legend_32 = fig2.legend(handles=legend_elements_32, loc='center',
                           bbox_to_anchor=(0.5, 0.37), framealpha=0.9, fontsize=10, ncol=5)

    # Legend for 72-site (Row 3)
    legend_elements_72 = []
    colors_72 = {
        'THQ': '#2ca02c', 'HHTP': '#d62728', 'HHTT': '#1f77b4', 'HHTN': '#ff7f0e'
    }
    for linker in TYPES_72:
        length = LENGTHS_72.get(linker, 0)
        legend_elements_72.append(Line2D([0], [0], marker='o', color='w',
                                     markerfacecolor=colors_72[linker],
                                     markersize=12,
                                     label=f'{linker} ({length:.1f}Å)',
                                     markeredgecolor='black', markeredgewidth=1))
    legend_elements_72.append(Line2D([0], [0], color='#333333', linewidth=2.5,
                                 label='Topological', alpha=0.9))
    legend_elements_72.append(Line2D([0], [0], color='#CCCCCC', linewidth=1.0,
                                 label='Spatial', alpha=0.5))

    legend_72 = fig2.legend(handles=legend_elements_72, loc='center',
                           bbox_to_anchor=(0.5, 0.07), framealpha=0.9, fontsize=11, ncol=6)



    plt.tight_layout(rect=[0.08, 0.02, 1, 0.98])

    output2 = 'Configurations_SA_vs_PT.png'
    plt.savefig(output2, dpi=300, bbox_inches='tight')
    print(f"  Saved: {output2}")
    plt.close(fig2)

    print(f"\n[Figures saved]")
    print(f"  Trajectories: {output1}")
    print(f"  Configurations: {output2}")

    # Summary
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    print(f"16-Site:")
    print(f"  SA Energy:  {hist16_sa[-1]:.4f}")
    print(f"  PT Energy:  {hist16_pt[-1]:.4f}")
    print(f"  Improvement: {100*(hist16_sa[-1]-hist16_pt[-1])/hist16_sa[-1]:.2f}%")
    print()
    print(f"32-Site:")
    print(f"  SA Energy:  {hist32_sa[-1]:.4f}")
    print(f"  PT Energy:  {hist32_pt[-1]:.4f}")
    print(f"  Improvement: {100*(hist32_sa[-1]-hist32_pt[-1])/hist32_sa[-1]:.2f}%")
    print()
    print(f"72-Site:")
    print(f"  SA Energy:  {hist72_sa[-1]:.4f}")
    print(f"  PT Energy:  {hist72_pt[-1]:.4f}")
    print(f"  Improvement: {100*(hist72_sa[-1]-hist72_pt[-1])/hist72_sa[-1]:.2f}%")
    print("="*70)

if __name__ == "__main__":
    plot_comparison_figure()

Energy Trajectory Comparison: SA vs Parallel Tempering

[16-Site System - 4 Linker Types]
  Graph: 16 nodes, 50 edges
  Running SA...
    Time: 0.35s | Final Energy: 85.4319
  Running PT...
    Time: 5.42s | Final Energy: 85.4319

[32-Site System - 8 Linker Types]
  Graph: 32 nodes, 222 edges
  Running SA...
    Time: 0.54s | Final Energy: 1616.8684
  Running PT...
    Time: 7.65s | Final Energy: 1616.8684

[72-Site System - 4 Linker Types]
  Graph: 72 nodes, 621 edges
  Running SA...
    Time: 0.42s | Final Energy: 4418.3263
  Running PT...
    Time: 6.83s | Final Energy: 4438.8794

[Generating energy trajectory comparison...]
  Saved: Energy_Trajectories_SA_vs_PT.png

[Generating configuration comparison...]


/tmp/ipython-input-2624528597.py:640: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.08, 0.02, 1, 0.98])


  Saved: Configurations_SA_vs_PT.png

[Figures saved]
  Trajectories: Energy_Trajectories_SA_vs_PT.png
  Configurations: Configurations_SA_vs_PT.png

SUMMARY
16-Site:
  SA Energy:  85.4319
  PT Energy:  85.4319
  Improvement: 0.00%

32-Site:
  SA Energy:  1616.8684
  PT Energy:  1616.8684
  Improvement: 0.00%

72-Site:
  SA Energy:  4418.3263
  PT Energy:  4438.8794
  Improvement: -0.47%


# Table 1

In [ ]:
# 16-Site Cu-THQ-HHTP Benchmark
# Correct bipartite detection + Statistics + Visualization

import networkx as nx
import numpy as np
import time
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
SEED = 42
RUNS = 100
STEPS = 20000

# Material Parameters (α = 0.01 as per Kang et al.)
MATERIAL_PARAMS = {
    'topological': {'alpha': 1.0, 'distance': 3.0},
    'spatial': {'alpha': 0.01, 'distance': 5.2}
}

LINKER_TYPES = ['THQ', 'HHTP']
LINKER_COUNTS = {'THQ': 8, 'HHTP': 8}
LINKER_LENGTHS = {'THQ': 2.42, 'HHTP': 4.87}

def add_edge_weights_to_graph_FIXED(G, params):
    """Distance-based spatial edge detection"""
    t_p = params['topological']
    s_p = params['spatial']
    w_topo = t_p['distance'] ** t_p['alpha']
    w_spatial = s_p['distance'] ** s_p['alpha']

    print(f"\n[Graph Construction]")
    print(f"  Topological Weight: {w_topo:.4f}")
    print(f"  Spatial Weight: {w_spatial:.4f}")

    pos = nx.get_node_attributes(G, 'pos')
    if not pos:
        pos = {node: data['pos'] for node, data in G.nodes(data=True) if 'pos' in data}

    topo_distances = []
    for u, v in G.edges():
        G[u][v]['weight'] = w_topo
        G[u][v]['edge_type'] = 'topological'
        pos_u = np.array(pos[u])
        pos_v = np.array(pos[v])
        dist = np.linalg.norm(pos_u - pos_v)
        topo_distances.append(dist)

    avg_topo_dist = np.mean(topo_distances)
    spatial_distance_min = 1.4 * avg_topo_dist
    spatial_distance_max = 2.1 * avg_topo_dist

    nodes = list(G.nodes())
    spatial_edges = set()

    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            if G.has_edge(u, v):
                continue
            pos_u = np.array(pos[u])
            pos_v = np.array(pos[v])
            dist = np.linalg.norm(pos_u - pos_v)
            if spatial_distance_min <= dist <= spatial_distance_max:
                spatial_edges.add((u, v))

    for u, v in spatial_edges:
        G.add_edge(u, v, weight=w_spatial, edge_type='spatial')

    topo_count = sum(1 for _, _, d in G.edges(data=True) if d.get('edge_type') == 'topological')
    spatial_count = sum(1 for _, _, d in G.edges(data=True) if d.get('edge_type') == 'spatial')
    print(f"  Topological edges: {topo_count}")
    print(f"  Spatial edges: {spatial_count}")
    print(f"  Total edges: {len(G.edges())}")
    return G

class MTVSolver:
    def __init__(self, graph, target_counts, linker_lengths, original_graph=None):
        if original_graph is not None and nx.is_bipartite(original_graph):
            color = nx.coloring.greedy_color(original_graph, strategy='largest_first')
            bipartite_set_0_original = {node for node, c in color.items() if c == 0}
            bipartite_set_1_original = {node for node, c in color.items() if c == 1}
            print(f"\n[Bipartite Structure - Original Graph]")
            print(f"  Sublattice A: {len(bipartite_set_0_original)} nodes")
            print(f"  Sublattice B: {len(bipartite_set_1_original)} nodes")
        else:
            bipartite_set_0_original = None
            bipartite_set_1_original = None

        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        self.graph = nx.relabel_nodes(graph, mapping)

        if bipartite_set_0_original is not None:
            self.bipartite_A = {mapping[node] for node in bipartite_set_0_original}
            self.bipartite_B = {mapping[node] for node in bipartite_set_1_original}
            print(f"\n[Bipartite Structure - After Relabeling]")
            print(f"  Sublattice A: {sorted(self.bipartite_A)}")
            print(f"  Sublattice B: {sorted(self.bipartite_B)}")
        else:
            self.bipartite_A = None
            self.bipartite_B = None

        self.edges = np.array(list(self.graph.edges()), dtype=int)
        self.linkers = list(target_counts.keys())
        self.lengths = linker_lengths
        self.length_lookup = np.array([linker_lengths[l] for l in self.linkers])
        self.edge_weights = np.array([self.graph[i][j].get('weight', 1.0) for i, j in self.edges])
        self.edge_types = np.array([self.graph[i][j].get('edge_type', 'unknown') for i, j in self.edges])

        self.state_pool = []
        for linker_idx, linker_name in enumerate(self.linkers):
            self.state_pool.extend([linker_idx] * target_counts[linker_name])
        self.state_pool = np.array(self.state_pool)

    def get_energy(self, current_state):
        u = self.edges[:, 0]
        v = self.edges[:, 1]
        len_u = self.length_lookup[current_state[u]]
        len_v = self.length_lookup[current_state[v]]
        edge_lengths = len_u + len_v
        mean_length = np.mean(edge_lengths)
        return np.sum(self.edge_weights * (edge_lengths - mean_length) ** 2)

    def create_perfect_alternation_state(self):
        if self.bipartite_A is None or self.bipartite_B is None:
            return np.array([i % 2 for i in range(len(self.state_pool))])
        state = np.zeros(len(self.state_pool), dtype=int)
        for node in self.bipartite_A:
            state[node] = 0
        for node in self.bipartite_B:
            state[node] = 1
        return state

    def analyze_state(self, state, label="State"):
        u = self.edges[:, 0]
        v = self.edges[:, 1]
        topo_mask = self.edge_types == 'topological'
        topo_edges = self.edges[topo_mask]
        alternating_count = sum(1 for i, j in topo_edges if state[i] != state[j])
        total_topo = len(topo_edges)

        len_u = self.length_lookup[state[u]]
        len_v = self.length_lookup[state[v]]
        edge_lengths = len_u + len_v
        mean_length = np.mean(edge_lengths)
        deviations = (edge_lengths - mean_length) ** 2
        topo_energy = np.sum(self.edge_weights[topo_mask] * deviations[topo_mask])
        spatial_energy = np.sum(self.edge_weights[~topo_mask] * deviations[~topo_mask])
        total_energy = topo_energy + spatial_energy

        print(f"\n[{label}]")
        print(f"  State: {state}")
        print(f"  Alternation: {alternating_count}/{total_topo} topological ({100*alternating_count/total_topo:.1f}%)")
        print(f"  Energy: Total={total_energy:.6f} (Topo={topo_energy:.4f}, Spatial={spatial_energy:.4f})")
        return alternating_count, total_topo, total_energy

    def solve(self, steps=100000, initial_temp=100.0, return_state=False):
        current_state = self.state_pool.copy()
        np.random.shuffle(current_state)
        current_energy = self.get_energy(current_state)
        best_state = current_state.copy()
        best_energy = current_energy
        n_sites = len(current_state)

        for i in range(steps):
            idx1, idx2 = np.random.randint(0, n_sites, 2)
            if current_state[idx1] == current_state[idx2]:
                continue
            current_state[idx1], current_state[idx2] = current_state[idx2], current_state[idx1]
            new_energy = self.get_energy(current_state)
            delta = new_energy - current_energy
            temp = initial_temp * (0.95 ** (i / 100))
            if delta < 0 or np.random.rand() < np.exp(-delta / (temp + 1e-10)):
                current_energy = new_energy
                if current_energy < best_energy:
                    best_energy = current_energy
                    best_state = current_state.copy()
            else:
                current_state[idx1], current_state[idx2] = current_state[idx2], current_state[idx1]

        if return_state:
            return best_energy, best_state
        return best_energy

def visualize_configuration(state, graph, linkers, title="Configuration", filename=None, show_energy=None):
    fig, ax = plt.subplots(figsize=(10, 8))
    if not all(isinstance(node, int) for node in graph.nodes()):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        graph = nx.relabel_nodes(graph, mapping)

    pos = nx.get_node_attributes(graph, 'pos')
    if not pos:
        pos = nx.spring_layout(graph, seed=SEED)

    pos_array = np.array(list(pos.values()))
    pos_min = pos_array.min(axis=0)
    pos_max = pos_array.max(axis=0)
    pos_range = pos_max - pos_min
    pos_normalized = {node: (p - pos_min) / pos_range for node, p in pos.items()}

    for i, j, data in graph.edges(data=True):
        edge_type = data.get('edge_type', 'unknown')
        x_coords = [pos_normalized[i][0], pos_normalized[j][0]]
        y_coords = [pos_normalized[i][1], pos_normalized[j][1]]
        if edge_type == 'topological':
            ax.plot(x_coords, y_coords, color='#333333', linewidth=2.5, alpha=0.9, zorder=5)
        else:
            ax.plot(x_coords, y_coords, color='#BBBBBB', linewidth=0.8, alpha=0.5, zorder=1)

    colors = {'THQ': '#2ca02c', 'HHTP': '#d62728'}
    for node_idx in graph.nodes():
        x, y = pos_normalized[node_idx]
        linker = linkers[state[node_idx]]
        color = colors.get(linker, 'gray')
        circle = plt.Circle((x, y), 0.04, color=color, alpha=0.95, ec='black', linewidth=2, zorder=10)
        ax.add_patch(circle)
        ax.text(x, y, linker[0], ha='center', va='center', fontsize=10, fontweight='bold', color='white', zorder=11)

    ax.set_aspect('equal')
    ax.axis('off')
    title_text = f'{title}\nEnergy = {show_energy:.6f}' if show_energy is not None else title
    ax.set_title(title_text, fontsize=14, fontweight='bold', pad=20)

    legend_elements = [
        Patch(facecolor='#2ca02c', edgecolor='black', label='THQ (2.42Å)'),
        Patch(facecolor='#d62728', edgecolor='black', label='HHTP (4.87Å)'),
        Line2D([0], [0], color='#333333', linewidth=2.5, label='Topological'),
        Line2D([0], [0], color='#BBBBBB', linewidth=0.8, label='Spatial'),
    ]
    ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=4, fontsize=9, frameon=True, fancybox=True, shadow=True)
    plt.tight_layout()

    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"  Saved: {filename}")
        plt.close(fig)
    else:
        plt.show()

def run_proper_bipartite_benchmark():
    np.random.seed(SEED)
    print("="*70)
    print(f"16-Site Cu-THQ-HHTP Benchmark")
    print("="*70)

    G16_original = nx.hexagonal_lattice_graph(2, 4, periodic=True)
    G16 = G16_original.copy()
    G16 = add_edge_weights_to_graph_FIXED(G16, MATERIAL_PARAMS)
    solver = MTVSolver(G16, LINKER_COUNTS, LINKER_LENGTHS, original_graph=G16_original)

    print("\n" + "="*70)
    print("TEST: Perfect Alternation")
    print("="*70)
    perfect_state = solver.create_perfect_alternation_state()
    alt_count, total, perfect_energy = solver.analyze_state(perfect_state, "Perfect Alternation")
    if alt_count == total:
        print("✓ TRUE PERFECT ALTERNATION: 100%")

    print("\n" + "="*70)
    print(f"OPTIMIZATION ({RUNS} runs, {STEPS} steps)")
    print("="*70)

    energies = []
    states = []
    times = []
    for i in range(RUNS):
        t0 = time.time()
        e, state = solver.solve(steps=STEPS, return_state=True)
        t1 = time.time()
        times.append(t1 - t0)
        energies.append(e)
        states.append(state.copy())
        if (i + 1) % 10 == 0:
            print(f"  Completed {i+1}/{RUNS} runs")

    min_e = min(energies)
    success = sum(1 for e in energies if abs(e - min_e) < 0.001)
    rate = success / RUNS
    avg_time = np.mean(times)

    print("\n" + "="*70)
    print("RESULTS")
    print("="*70)
    print(f"Perfect Alternation Energy: {perfect_energy:.6f}")
    print(f"SA Minimum Energy:          {min_e:.6f}")
    print(f"Energy Gap:                 {abs(min_e - perfect_energy):.6f}")

    if abs(min_e - perfect_energy) < 0.001:
        print("\n✓✓✓ SUCCESS: SA FOUND PERFECT ALTERNATION!")
    else:
        gap_pct = 100 * abs(min_e - perfect_energy) / perfect_energy
        print(f"\n✗ SA did not find perfect alternation ({gap_pct:.2f}% gap)")

    print(f"\nSuccess Rate:     {rate*100:.1f}%")
    print(f"Avg Time per Run: {avg_time:.4f}s")

    if rate > 0:
        if rate >= 0.99:
            runs_needed = 1
        else:
            runs_needed = math.ceil(math.log(0.01) / math.log(1 - rate))
        print(f"Runs for 99% Conf: {runs_needed}")
        print(f"Total Time-to-Solution: {runs_needed * avg_time:.2f}s")
    else:
        print("Runs for 99% Conf: N/A")
        print("Total Time-to-Solution: N/A")

    best_idx = energies.index(min_e)
    best_state = states[best_idx]
    print("\n" + "="*70)
    print("SA BEST STATE")
    print("="*70)
    solver.analyze_state(best_state, "SA Best")

    print("\n" + "="*70)
    print("GENERATING VISUALIZATION")
    print("="*70)
    visualize_configuration(best_state, solver.graph, LINKER_TYPES, title="Lowest Energy Configuration", filename="lowest_energy_16site.png", show_energy=min_e)

    if abs(min_e - perfect_energy) > 0.001:
        print("\n" + "="*70)
        print("HYPERPARAMETER SUGGESTIONS")
        print("="*70)
        print("Try:")
        print("  1. STEPS = 500000")
        print("  2. Cooling: 0.99 instead of 0.95")
        print("  3. initial_temp = 500")

if __name__ == "__main__":
    run_proper_bipartite_benchmark()

16-Site Cu-THQ-HHTP Benchmark

[Graph Construction]
  Topological Weight: 3.0000
  Spatial Weight: 1.0166
  Topological edges: 24
  Spatial edges: 26
  Total edges: 50

[Bipartite Structure - Original Graph]
  Sublattice A: 8 nodes
  Sublattice B: 8 nodes

[Bipartite Structure - After Relabeling]
  Sublattice A: [0, 2, 5, 7, 8, 10, 13, 15]
  Sublattice B: [1, 3, 4, 6, 9, 11, 12, 14]

TEST: Perfect Alternation

[Perfect Alternation]
  State: [0 1 0 1 1 0 1 0 0 1 0 1 1 0 1 0]
  Alternation: 24/24 topological (100.0%)
  Energy: Total=85.431934 (Topo=0.0000, Spatial=85.4319)
✓ TRUE PERFECT ALTERNATION: 100%

OPTIMIZATION (100 runs, 20000 steps)
  Completed 10/100 runs
  Completed 20/100 runs
  Completed 30/100 runs
  Completed 40/100 runs
  Completed 50/100 runs
  Completed 60/100 runs
  Completed 70/100 runs
  Completed 80/100 runs
  Completed 90/100 runs
  Completed 100/100 runs

RESULTS
Perfect Alternation Energy: 85.431934
SA Minimum Energy:          85.431934
Energy Gap:              

In [ ]:
# 32-Site Benchmark - 8 Linker Types
# Correct bipartite detection + Statistics + Visualization

import networkx as nx
import numpy as np
import time
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
SEED = 42
RUNS = 100
STEPS = 20000

# Material Parameters
MATERIAL_PARAMS = {
    'topological': {'alpha': 1.0, 'distance': 3.0},
    'spatial': {'alpha': 0.01, 'distance': 5.2}
}

# 32-Site: 8 Linker Types (4 of each)
TYPES_32 = ['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8']
COUNTS_32 = {t: 4 for t in TYPES_32}
LENGTHS_32 = {
    'L1': 2.0, 'L2': 3.0, 'L3': 4.0, 'L4': 5.0,
    'L5': 6.0, 'L6': 7.0, 'L7': 8.0, 'L8': 9.0
}

def add_edge_weights_to_graph(G, params):
    """Distance-based spatial edge detection"""
    t_p = params['topological']
    s_p = params['spatial']
    w_topo = t_p['distance'] ** t_p['alpha']
    w_spatial = s_p['distance'] ** s_p['alpha']

    print(f"\n[Graph Construction]")
    print(f"  Topological Weight: {w_topo:.4f}")
    print(f"  Spatial Weight: {w_spatial:.4f}")

    pos = nx.get_node_attributes(G, 'pos')
    if not pos:
        pos = {node: data['pos'] for node, data in G.nodes(data=True) if 'pos' in data}

    topo_distances = []
    for u, v in G.edges():
        G[u][v]['weight'] = w_topo
        G[u][v]['edge_type'] = 'topological'
        pos_u = np.array(pos[u])
        pos_v = np.array(pos[v])
        dist = np.linalg.norm(pos_u - pos_v)
        topo_distances.append(dist)

    avg_topo_dist = np.mean(topo_distances)
    spatial_distance_min = 1.4 * avg_topo_dist
    spatial_distance_max = 2.1 * avg_topo_dist

    nodes = list(G.nodes())
    spatial_edges = set()

    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            if G.has_edge(u, v):
                continue
            pos_u = np.array(pos[u])
            pos_v = np.array(pos[v])
            dist = np.linalg.norm(pos_u - pos_v)
            if spatial_distance_min <= dist <= spatial_distance_max:
                spatial_edges.add((u, v))

    for u, v in spatial_edges:
        G.add_edge(u, v, weight=w_spatial, edge_type='spatial')

    topo_count = sum(1 for _, _, d in G.edges(data=True) if d.get('edge_type') == 'topological')
    spatial_count = sum(1 for _, _, d in G.edges(data=True) if d.get('edge_type') == 'spatial')
    print(f"  Topological edges: {topo_count}")
    print(f"  Spatial edges: {spatial_count}")
    print(f"  Total edges: {len(G.edges())}")
    return G

class MTVSolver:
    def __init__(self, graph, target_counts, linker_lengths, original_graph=None):
        if original_graph is not None and nx.is_bipartite(original_graph):
            color = nx.coloring.greedy_color(original_graph, strategy='largest_first')
            bipartite_set_0_original = {node for node, c in color.items() if c == 0}
            bipartite_set_1_original = {node for node, c in color.items() if c == 1}
            print(f"\n[Bipartite Structure Detected]")
            print(f"  Sublattice A: {len(bipartite_set_0_original)} nodes")
            print(f"  Sublattice B: {len(bipartite_set_1_original)} nodes")
        else:
            bipartite_set_0_original = None
            bipartite_set_1_original = None

        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        self.graph = nx.relabel_nodes(graph, mapping)

        if bipartite_set_0_original is not None:
            self.bipartite_A = {mapping[node] for node in bipartite_set_0_original}
            self.bipartite_B = {mapping[node] for node in bipartite_set_1_original}
        else:
            self.bipartite_A = None
            self.bipartite_B = None

        self.edges = np.array(list(self.graph.edges()), dtype=int)
        self.linkers = list(target_counts.keys())
        self.lengths = linker_lengths
        self.length_lookup = np.array([linker_lengths[l] for l in self.linkers])
        self.edge_weights = np.array([self.graph[i][j].get('weight', 1.0) for i, j in self.edges])
        self.edge_types = np.array([self.graph[i][j].get('edge_type', 'unknown') for i, j in self.edges])

        self.state_pool = []
        for linker_idx, linker_name in enumerate(self.linkers):
            self.state_pool.extend([linker_idx] * target_counts[linker_name])
        self.state_pool = np.array(self.state_pool)

    def get_energy(self, current_state):
        u = self.edges[:, 0]
        v = self.edges[:, 1]
        len_u = self.length_lookup[current_state[u]]
        len_v = self.length_lookup[current_state[v]]
        edge_lengths = len_u + len_v
        mean_length = np.mean(edge_lengths)
        return np.sum(self.edge_weights * (edge_lengths - mean_length) ** 2)

    def solve(self, steps=100000, initial_temp=100.0, return_state=False):
        current_state = self.state_pool.copy()
        np.random.shuffle(current_state)
        current_energy = self.get_energy(current_state)
        best_state = current_state.copy()
        best_energy = current_energy
        n_sites = len(current_state)

        for i in range(steps):
            idx1, idx2 = np.random.randint(0, n_sites, 2)
            if current_state[idx1] == current_state[idx2]:
                continue
            current_state[idx1], current_state[idx2] = current_state[idx2], current_state[idx1]
            new_energy = self.get_energy(current_state)
            delta = new_energy - current_energy
            temp = initial_temp * (0.95 ** (i / 100))
            if delta < 0 or np.random.rand() < np.exp(-delta / (temp + 1e-10)):
                current_energy = new_energy
                if current_energy < best_energy:
                    best_energy = current_energy
                    best_state = current_state.copy()
            else:
                current_state[idx1], current_state[idx2] = current_state[idx2], current_state[idx1]

        if return_state:
            return best_energy, best_state
        return best_energy

def visualize_configuration(state, graph, linkers, title="Configuration", filename=None, show_energy=None):
    fig, ax = plt.subplots(figsize=(12, 10))
    if not all(isinstance(node, int) for node in graph.nodes()):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        graph = nx.relabel_nodes(graph, mapping)

    pos = nx.get_node_attributes(graph, 'pos')
    if not pos:
        pos = nx.spring_layout(graph, seed=SEED)

    pos_array = np.array(list(pos.values()))
    pos_min = pos_array.min(axis=0)
    pos_max = pos_array.max(axis=0)
    pos_range = pos_max - pos_min
    pos_normalized = {node: (p - pos_min) / pos_range for node, p in pos.items()}

    for i, j, data in graph.edges(data=True):
        edge_type = data.get('edge_type', 'unknown')
        x_coords = [pos_normalized[i][0], pos_normalized[j][0]]
        y_coords = [pos_normalized[i][1], pos_normalized[j][1]]
        if edge_type == 'topological':
            ax.plot(x_coords, y_coords, color='#333333', linewidth=2.0, alpha=0.8, zorder=5)
        else:
            ax.plot(x_coords, y_coords, color='#CCCCCC', linewidth=0.5, alpha=0.3, zorder=1)

    # Use colormap for 8 linker types
    colors_map = plt.cm.tab10(np.linspace(0, 1, len(set(linkers))))
    colors = {linker: colors_map[i] for i, linker in enumerate(sorted(set(linkers)))}

    for node_idx in graph.nodes():
        x, y = pos_normalized[node_idx]
        linker = linkers[state[node_idx]]
        color = colors.get(linker, 'gray')
        circle = plt.Circle((x, y), 0.03, color=color, alpha=0.95, ec='black', linewidth=1.5, zorder=10)
        ax.add_patch(circle)
        ax.text(x, y, linker, ha='center', va='center', fontsize=6, fontweight='bold', color='white', zorder=11)

    ax.set_aspect('equal')
    ax.axis('off')
    title_text = f'{title}\nEnergy = {show_energy:.6f}' if show_energy is not None else title
    ax.set_title(title_text, fontsize=14, fontweight='bold', pad=20)

    legend_elements = [Patch(facecolor=colors[l], edgecolor='black', label=f'{l} ({LENGTHS_32[l]:.1f}Å)') for l in sorted(set(linkers))]
    ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4, fontsize=8, frameon=True)
    plt.tight_layout()

    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"  Saved: {filename}")
        plt.close(fig)

def run_32site_benchmark():
    np.random.seed(SEED)
    print("="*70)
    print(f"32-Site Benchmark (8 Linker Types)")
    print("="*70)

    # 4x4 periodic honeycomb = 32 nodes
    G32_original = nx.hexagonal_lattice_graph(4, 4, periodic=True)
    G32 = G32_original.copy()
    G32 = add_edge_weights_to_graph(G32, MATERIAL_PARAMS)
    solver = MTVSolver(G32, COUNTS_32, LENGTHS_32, original_graph=G32_original)

    print("\n" + "="*70)
    print(f"OPTIMIZATION ({RUNS} runs, {STEPS} steps)")
    print("="*70)

    energies = []
    states = []
    times = []
    for i in range(RUNS):
        t0 = time.time()
        e, state = solver.solve(steps=STEPS, return_state=True)
        t1 = time.time()
        times.append(t1 - t0)
        energies.append(e)
        states.append(state.copy())
        if (i + 1) % 10 == 0:
            print(f"  Completed {i+1}/{RUNS} runs")

    min_e = min(energies)
    success = sum(1 for e in energies if abs(e - min_e) < 0.001)
    rate = success / RUNS
    avg_time = np.mean(times)

    print("\n" + "="*70)
    print("RESULTS")
    print("="*70)
    print(f"Min Energy Found:   {min_e:.6f}")
    print(f"Success Rate:       {rate*100:.1f}%")
    print(f"Avg Time per Run:   {avg_time:.4f}s")

    if rate > 0:
        if rate >= 0.99:
            runs_needed = 1
        else:
            runs_needed = math.ceil(math.log(0.01) / math.log(1 - rate))
        print(f"Runs for 99% Conf:  {runs_needed}")
        print(f"Total Time-to-Solution: {runs_needed * avg_time:.2f}s")
    else:
        print("Runs for 99% Conf:  N/A")
        print("Total Time-to-Solution: N/A")

    best_idx = energies.index(min_e)
    best_state = states[best_idx]

    print("\n" + "="*70)
    print("GENERATING VISUALIZATION")
    print("="*70)
    visualize_configuration(best_state, solver.graph, TYPES_32, title="Lowest Energy Configuration (32-Site)", filename="lowest_energy_32site.png", show_energy=min_e)
    print("="*70)

if __name__ == "__main__":
    run_32site_benchmark()

32-Site Benchmark (8 Linker Types)

[Graph Construction]
  Topological Weight: 3.0000
  Spatial Weight: 1.0166
  Topological edges: 48
  Spatial edges: 174
  Total edges: 222

[Bipartite Structure Detected]
  Sublattice A: 16 nodes
  Sublattice B: 16 nodes

OPTIMIZATION (100 runs, 20000 steps)
  Completed 10/100 runs
  Completed 20/100 runs
  Completed 30/100 runs
  Completed 40/100 runs
  Completed 50/100 runs
  Completed 60/100 runs
  Completed 70/100 runs
  Completed 80/100 runs
  Completed 90/100 runs
  Completed 100/100 runs

RESULTS
Min Energy Found:   1616.868367
Success Rate:       46.0%
Avg Time per Run:   0.9076s
Runs for 99% Conf:  8
Total Time-to-Solution: 7.26s

GENERATING VISUALIZATION
  Saved: lowest_energy_32site.png


In [ ]:
# 72-Site Benchmark - 4 Linker Types
# Correct bipartite detection + Statistics + Visualization

import networkx as nx
import numpy as np
import time
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
SEED = 42
RUNS = 100
STEPS = 20000

# Material Parameters
MATERIAL_PARAMS = {
    'topological': {'alpha': 1.0, 'distance': 3.0},
    'spatial': {'alpha': 0.01, 'distance': 5.2}
}

# 72-Site: 4 Linker Types (18 of each)
TYPES_72 = ['THQ', 'HHTP', 'HHTT', 'HHTN']
COUNTS_72 = {t: 18 for t in TYPES_72}
LENGTHS_72 = {
    'THQ': 2.0, 'HHTP': 4.0, 'HHTT': 6.0, 'HHTN': 8.0
}

def add_edge_weights_to_graph(G, params):
    """Distance-based spatial edge detection"""
    t_p = params['topological']
    s_p = params['spatial']
    w_topo = t_p['distance'] ** t_p['alpha']
    w_spatial = s_p['distance'] ** s_p['alpha']

    print(f"\n[Graph Construction]")
    print(f"  Topological Weight: {w_topo:.4f}")
    print(f"  Spatial Weight: {w_spatial:.4f}")

    pos = nx.get_node_attributes(G, 'pos')
    if not pos:
        pos = {node: data['pos'] for node, data in G.nodes(data=True) if 'pos' in data}

    topo_distances = []
    for u, v in G.edges():
        G[u][v]['weight'] = w_topo
        G[u][v]['edge_type'] = 'topological'
        pos_u = np.array(pos[u])
        pos_v = np.array(pos[v])
        dist = np.linalg.norm(pos_u - pos_v)
        topo_distances.append(dist)

    avg_topo_dist = np.mean(topo_distances)
    spatial_distance_min = 1.4 * avg_topo_dist
    spatial_distance_max = 2.1 * avg_topo_dist

    nodes = list(G.nodes())
    spatial_edges = set()

    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            if G.has_edge(u, v):
                continue
            pos_u = np.array(pos[u])
            pos_v = np.array(pos[v])
            dist = np.linalg.norm(pos_u - pos_v)
            if spatial_distance_min <= dist <= spatial_distance_max:
                spatial_edges.add((u, v))

    for u, v in spatial_edges:
        G.add_edge(u, v, weight=w_spatial, edge_type='spatial')

    topo_count = sum(1 for _, _, d in G.edges(data=True) if d.get('edge_type') == 'topological')
    spatial_count = sum(1 for _, _, d in G.edges(data=True) if d.get('edge_type') == 'spatial')
    print(f"  Topological edges: {topo_count}")
    print(f"  Spatial edges: {spatial_count}")
    print(f"  Total edges: {len(G.edges())}")
    return G

class MTVSolver:
    def __init__(self, graph, target_counts, linker_lengths, original_graph=None):
        if original_graph is not None and nx.is_bipartite(original_graph):
            color = nx.coloring.greedy_color(original_graph, strategy='largest_first')
            bipartite_set_0_original = {node for node, c in color.items() if c == 0}
            bipartite_set_1_original = {node for node, c in color.items() if c == 1}
            print(f"\n[Bipartite Structure Detected]")
            print(f"  Sublattice A: {len(bipartite_set_0_original)} nodes")
            print(f"  Sublattice B: {len(bipartite_set_1_original)} nodes")
        else:
            bipartite_set_0_original = None
            bipartite_set_1_original = None

        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        self.graph = nx.relabel_nodes(graph, mapping)

        if bipartite_set_0_original is not None:
            self.bipartite_A = {mapping[node] for node in bipartite_set_0_original}
            self.bipartite_B = {mapping[node] for node in bipartite_set_1_original}
        else:
            self.bipartite_A = None
            self.bipartite_B = None

        self.edges = np.array(list(self.graph.edges()), dtype=int)
        self.linkers = list(target_counts.keys())
        self.lengths = linker_lengths
        self.length_lookup = np.array([linker_lengths[l] for l in self.linkers])
        self.edge_weights = np.array([self.graph[i][j].get('weight', 1.0) for i, j in self.edges])
        self.edge_types = np.array([self.graph[i][j].get('edge_type', 'unknown') for i, j in self.edges])

        self.state_pool = []
        for linker_idx, linker_name in enumerate(self.linkers):
            self.state_pool.extend([linker_idx] * target_counts[linker_name])
        self.state_pool = np.array(self.state_pool)

    def get_energy(self, current_state):
        u = self.edges[:, 0]
        v = self.edges[:, 1]
        len_u = self.length_lookup[current_state[u]]
        len_v = self.length_lookup[current_state[v]]
        edge_lengths = len_u + len_v
        mean_length = np.mean(edge_lengths)
        return np.sum(self.edge_weights * (edge_lengths - mean_length) ** 2)

    def solve(self, steps=100000, initial_temp=100.0, return_state=False):
        current_state = self.state_pool.copy()
        np.random.shuffle(current_state)
        current_energy = self.get_energy(current_state)
        best_state = current_state.copy()
        best_energy = current_energy
        n_sites = len(current_state)

        for i in range(steps):
            idx1, idx2 = np.random.randint(0, n_sites, 2)
            if current_state[idx1] == current_state[idx2]:
                continue
            current_state[idx1], current_state[idx2] = current_state[idx2], current_state[idx1]
            new_energy = self.get_energy(current_state)
            delta = new_energy - current_energy
            temp = initial_temp * (0.95 ** (i / 100))
            if delta < 0 or np.random.rand() < np.exp(-delta / (temp + 1e-10)):
                current_energy = new_energy
                if current_energy < best_energy:
                    best_energy = current_energy
                    best_state = current_state.copy()
            else:
                current_state[idx1], current_state[idx2] = current_state[idx2], current_state[idx1]

        if return_state:
            return best_energy, best_state
        return best_energy

def visualize_configuration(state, graph, linkers, title="Configuration", filename=None, show_energy=None):
    fig, ax = plt.subplots(figsize=(14, 12))
    if not all(isinstance(node, int) for node in graph.nodes()):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        graph = nx.relabel_nodes(graph, mapping)

    pos = nx.get_node_attributes(graph, 'pos')
    if not pos:
        pos = nx.spring_layout(graph, seed=SEED)

    pos_array = np.array(list(pos.values()))
    pos_min = pos_array.min(axis=0)
    pos_max = pos_array.max(axis=0)
    pos_range = pos_max - pos_min
    pos_normalized = {node: (p - pos_min) / pos_range for node, p in pos.items()}

    for i, j, data in graph.edges(data=True):
        edge_type = data.get('edge_type', 'unknown')
        x_coords = [pos_normalized[i][0], pos_normalized[j][0]]
        y_coords = [pos_normalized[i][1], pos_normalized[j][1]]
        if edge_type == 'topological':
            ax.plot(x_coords, y_coords, color='#333333', linewidth=1.5, alpha=0.7, zorder=5)
        else:
            ax.plot(x_coords, y_coords, color='#DDDDDD', linewidth=0.3, alpha=0.3, zorder=1)

    # 4 linker types - use distinct colors
    colors = {
        'THQ': '#2ca02c',   # Green
        'HHTP': '#d62728',  # Red
        'HHTT': '#1f77b4',  # Blue
        'HHTN': '#ff7f0e'   # Orange
    }

    for node_idx in graph.nodes():
        x, y = pos_normalized[node_idx]
        linker = linkers[state[node_idx]]
        color = colors.get(linker, 'gray')
        circle = plt.Circle((x, y), 0.02, color=color, alpha=0.95, ec='black', linewidth=1, zorder=10)
        ax.add_patch(circle)

    ax.set_aspect('equal')
    ax.axis('off')
    title_text = f'{title}\nEnergy = {show_energy:.6f}' if show_energy is not None else title
    ax.set_title(title_text, fontsize=14, fontweight='bold', pad=20)

    legend_elements = [Patch(facecolor=colors[l], edgecolor='black', label=f'{l} ({LENGTHS_72[l]:.1f}Å)') for l in TYPES_72]
    ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4, fontsize=9, frameon=True)
    plt.tight_layout()

    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"  Saved: {filename}")
        plt.close(fig)

def run_72site_benchmark():
    np.random.seed(SEED)
    print("="*70)
    print(f"72-Site Benchmark (4 Linker Types)")
    print("="*70)

    # 6x6 periodic honeycomb = 72 nodes
    G72_original = nx.hexagonal_lattice_graph(6, 6, periodic=True)
    G72 = G72_original.copy()
    G72 = add_edge_weights_to_graph(G72, MATERIAL_PARAMS)
    solver = MTVSolver(G72, COUNTS_72, LENGTHS_72, original_graph=G72_original)

    print("\n" + "="*70)
    print(f"OPTIMIZATION ({RUNS} runs, {STEPS} steps)")
    print("="*70)

    energies = []
    states = []
    times = []
    for i in range(RUNS):
        t0 = time.time()
        e, state = solver.solve(steps=STEPS, return_state=True)
        t1 = time.time()
        times.append(t1 - t0)
        energies.append(e)
        states.append(state.copy())
        if (i + 1) % 10 == 0:
            print(f"  Completed {i+1}/{RUNS} runs")

    min_e = min(energies)
    success = sum(1 for e in energies if abs(e - min_e) < 0.001)
    rate = success / RUNS
    avg_time = np.mean(times)

    print("\n" + "="*70)
    print("RESULTS")
    print("="*70)
    print(f"Min Energy Found:   {min_e:.6f}")
    print(f"Success Rate:       {rate*100:.1f}%")
    print(f"Avg Time per Run:   {avg_time:.4f}s")

    if rate > 0:
        if rate >= 0.99:
            runs_needed = 1
        else:
            runs_needed = math.ceil(math.log(0.01) / math.log(1 - rate))
        print(f"Runs for 99% Conf:  {runs_needed}")
        print(f"Total Time-to-Solution: {runs_needed * avg_time:.2f}s")
    else:
        print("Runs for 99% Conf:  N/A")
        print("Total Time-to-Solution: N/A")

    best_idx = energies.index(min_e)
    best_state = states[best_idx]

    print("\n" + "="*70)
    print("GENERATING VISUALIZATION")
    print("="*70)
    visualize_configuration(best_state, solver.graph, TYPES_72, title="Lowest Energy Configuration (72-Site)", filename="lowest_energy_72site.png", show_energy=min_e)
    print("="*70)

if __name__ == "__main__":
    run_72site_benchmark()

72-Site Benchmark (4 Linker Types)

[Graph Construction]
  Topological Weight: 3.0000
  Spatial Weight: 1.0166
  Topological edges: 108
  Spatial edges: 513
  Total edges: 621

[Bipartite Structure Detected]
  Sublattice A: 36 nodes
  Sublattice B: 36 nodes

OPTIMIZATION (100 runs, 20000 steps)
  Completed 10/100 runs
  Completed 20/100 runs
  Completed 30/100 runs
  Completed 40/100 runs
  Completed 50/100 runs
  Completed 60/100 runs
  Completed 70/100 runs
  Completed 80/100 runs
  Completed 90/100 runs
  Completed 100/100 runs

RESULTS
Min Energy Found:   4369.684121
Success Rate:       3.0%
Avg Time per Run:   0.9340s
Runs for 99% Conf:  152
Total Time-to-Solution: 141.97s

GENERATING VISUALIZATION
  Saved: lowest_energy_72site.png


## Parallel Tempering (PT) Runs


In [ ]:
# 16-Site Benchmark - Parallel Tempering (Replica Exchange)
# 2 Linker Types (THQ, HHTP)

import networkx as nx
import numpy as np
import time
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
SEED = 42
RUNS = 10
TOTAL_MC_STEPS = 20000

# Parallel Tempering Parameters
NUM_REPLICAS = 16
T_MIN = 0.1
T_MAX = 100.0
EXCHANGE_INTERVAL = 100

# Material Parameters
MATERIAL_PARAMS = {
    'topological': {'alpha': 1.0, 'distance': 3.0},
    'spatial': {'alpha': 0.01, 'distance': 5.2}
}

# 16-Site: 2 Linker Types (8 of each)
TYPES_16 = ['THQ', 'HHTP']
COUNTS_16 = {'THQ': 8, 'HHTP': 8}
LENGTHS_16 = {'THQ': 2.42, 'HHTP': 4.87}

def add_edge_weights_to_graph(G, params):
    """Distance-based spatial edge detection"""
    t_p = params['topological']
    s_p = params['spatial']
    w_topo = t_p['distance'] ** t_p['alpha']
    w_spatial = s_p['distance'] ** s_p['alpha']

    pos = nx.get_node_attributes(G, 'pos')
    if not pos:
        pos = {node: data['pos'] for node, data in G.nodes(data=True) if 'pos' in data}

    topo_distances = []
    for u, v in G.edges():
        G[u][v]['weight'] = w_topo
        G[u][v]['edge_type'] = 'topological'
        pos_u = np.array(pos[u])
        pos_v = np.array(pos[v])
        dist = np.linalg.norm(pos_u - pos_v)
        topo_distances.append(dist)

    avg_topo_dist = np.mean(topo_distances)
    spatial_distance_min = 1.4 * avg_topo_dist
    spatial_distance_max = 2.1 * avg_topo_dist

    nodes = list(G.nodes())
    spatial_edges = set()

    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            if G.has_edge(u, v):
                continue
            pos_u = np.array(pos[u])
            pos_v = np.array(pos[v])
            dist = np.linalg.norm(pos_u - pos_v)
            if spatial_distance_min <= dist <= spatial_distance_max:
                spatial_edges.add((u, v))

    for u, v in spatial_edges:
        G.add_edge(u, v, weight=w_spatial, edge_type='spatial')

    return G

def get_bipartite_sets(G):
    """Get the two sublattices of the bipartite hexagonal lattice"""
    if nx.is_bipartite(G):
        color = nx.coloring.greedy_color(G, strategy='largest_first')
        set_0 = {node for node, c in color.items() if c == 0}
        set_1 = {node for node, c in color.items() if c == 1}
        return set_0, set_1
    else:
        return None, None

class ParallelTemperingSolver:
    def __init__(self, graph, target_counts, linker_lengths, original_graph=None):
        # Detect bipartite structure BEFORE relabeling
        if original_graph is not None and nx.is_bipartite(original_graph):
            color = nx.coloring.greedy_color(original_graph, strategy='largest_first')
            bipartite_set_0_original = {node for node, c in color.items() if c == 0}
            bipartite_set_1_original = {node for node, c in color.items() if c == 1}
            print(f"\n[Bipartite Structure Detected]")
            print(f"  Sublattice A: {len(bipartite_set_0_original)} nodes")
            print(f"  Sublattice B: {len(bipartite_set_1_original)} nodes")
        else:
            bipartite_set_0_original = None
            bipartite_set_1_original = None

        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        self.graph = nx.relabel_nodes(graph, mapping)

        # Map bipartite sets to new labels
        if bipartite_set_0_original is not None:
            self.bipartite_A = {mapping[node] for node in bipartite_set_0_original}
            self.bipartite_B = {mapping[node] for node in bipartite_set_1_original}
        else:
            self.bipartite_A = None
            self.bipartite_B = None

        self.edges = np.array(list(self.graph.edges()), dtype=int)
        self.linkers = list(target_counts.keys())
        self.lengths = linker_lengths
        self.length_lookup = np.array([linker_lengths[l] for l in self.linkers])
        self.edge_weights = np.array([self.graph[i][j].get('weight', 1.0) for i, j in self.edges])
        self.edge_types = np.array([self.graph[i][j].get('edge_type', 'unknown') for i, j in self.edges])

        self.state_pool = []
        for linker_idx, linker_name in enumerate(self.linkers):
            self.state_pool.extend([linker_idx] * target_counts[linker_name])
        self.state_pool = np.array(self.state_pool)
        self.n_sites = len(self.state_pool)

        # Temperature ladder (geometric spacing)
        self.temperatures = np.logspace(np.log10(T_MIN), np.log10(T_MAX), NUM_REPLICAS)
        self.betas = 1.0 / self.temperatures

        print(f"\n[Parallel Tempering Setup]")
        print(f"  Replicas: {NUM_REPLICAS}")
        print(f"  T_min: {T_MIN:.2f}, T_max: {T_MAX:.2f}")
        print(f"  Beta range: [{self.betas[0]:.4f}, {self.betas[-1]:.4f}]")

    def get_energy(self, current_state):
        u = self.edges[:, 0]
        v = self.edges[:, 1]
        len_u = self.length_lookup[current_state[u]]
        len_v = self.length_lookup[current_state[v]]
        edge_lengths = len_u + len_v
        mean_length = np.mean(edge_lengths)
        return np.sum(self.edge_weights * (edge_lengths - mean_length) ** 2)

    def analyze_state(self, state):
        """Analyze alternation pattern"""
        u = self.edges[:, 0]
        v = self.edges[:, 1]
        topo_mask = self.edge_types == 'topological'
        topo_edges = self.edges[topo_mask]
        alternating_count = sum(1 for i, j in topo_edges if state[i] != state[j])
        total_topo = len(topo_edges)
        return alternating_count, total_topo

    def solve(self, steps=TOTAL_MC_STEPS, return_state=False):
        # Initialize replicas - each gets independent random configuration
        replicas = []
        energies = []

        for _ in range(NUM_REPLICAS):
            state = self.state_pool.copy()
            np.random.shuffle(state)
            replicas.append(state.copy())
            energies.append(self.get_energy(state))

        # Track best solution across all replicas
        best_energy_global = min(energies)
        best_state_global = replicas[energies.index(best_energy_global)].copy()

        # Statistics tracking
        exchange_attempts = 0
        exchange_accepts = 0

        # Main loop
        for step in range(steps):
            # 1. Monte Carlo moves for each replica
            for i in range(NUM_REPLICAS):
                # Propose swap move
                idx1, idx2 = np.random.randint(0, self.n_sites, 2)
                if replicas[i][idx1] == replicas[i][idx2]:
                    continue

                current_E = energies[i]

                # Perform swap
                replicas[i][idx1], replicas[i][idx2] = replicas[i][idx2], replicas[i][idx1]
                new_E = self.get_energy(replicas[i])
                delta_E = new_E - current_E

                # Metropolis acceptance at temperature T_i
                beta = self.betas[i]
                if delta_E < 0 or np.random.rand() < np.exp(-beta * delta_E):
                    # Accept
                    energies[i] = new_E
                    if new_E < best_energy_global:
                        best_energy_global = new_E
                        best_state_global = replicas[i].copy()
                else:
                    # Reject - revert swap
                    replicas[i][idx1], replicas[i][idx2] = replicas[i][idx2], replicas[i][idx1]

            # 2. Replica exchange attempts
            if step % EXCHANGE_INTERVAL == 0 and step > 0:
                # Proper exchange protocol
                # Alternate between even (0-1, 2-3, ...) and odd (1-2, 3-4, ...) pairs
                offset = (step // EXCHANGE_INTERVAL) % 2

                for i in range(offset, NUM_REPLICAS - 1, 2):
                    j = i + 1

                    beta_i = self.betas[i]
                    beta_j = self.betas[j]
                    E_i = energies[i]
                    E_j = energies[j]

                    delta_beta = beta_i - beta_j
                    delta_E = E_j - E_i
                    log_acceptance = delta_beta * delta_E

                    exchange_attempts += 1

                    if log_acceptance >= 0 or np.random.rand() < np.exp(log_acceptance):
                        # Accept exchange - swap states
                        replicas[i], replicas[j] = replicas[j], replicas[i]
                        energies[i], energies[j] = energies[j], energies[i]
                        exchange_accepts += 1

        # Print exchange statistics
        if exchange_attempts > 0:
            exchange_rate = 100.0 * exchange_accepts / exchange_attempts
            print(f"  Exchange rate: {exchange_rate:.1f}% ({exchange_accepts}/{exchange_attempts})")

        if return_state:
            return best_energy_global, best_state_global
        return best_energy_global

def visualize_configuration(state, graph, linkers, title="Configuration", filename=None, show_energy=None):
    fig, ax = plt.subplots(figsize=(10, 8))

    if not all(isinstance(node, int) for node in graph.nodes()):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        graph = nx.relabel_nodes(graph, mapping)

    pos = nx.get_node_attributes(graph, 'pos')
    if not pos:
        pos = nx.spring_layout(graph, seed=SEED)

    pos_array = np.array(list(pos.values()))
    pos_min = pos_array.min(axis=0)
    pos_max = pos_array.max(axis=0)
    pos_range = pos_max - pos_min
    pos_normalized = {node: (p - pos_min) / pos_range for node, p in pos.items()}

    # Draw edges
    for i, j, data in graph.edges(data=True):
        edge_type = data.get('edge_type', 'unknown')
        x_coords = [pos_normalized[i][0], pos_normalized[j][0]]
        y_coords = [pos_normalized[i][1], pos_normalized[j][1]]

        if edge_type == 'topological':
            ax.plot(x_coords, y_coords, color='#333333', linewidth=2.5, alpha=0.9, zorder=5)
        else:
            ax.plot(x_coords, y_coords, color='#BBBBBB', linewidth=0.8, alpha=0.5, zorder=1)

    # Draw nodes with colors
    colors = {'THQ': '#2ca02c', 'HHTP': '#d62728'}  # Green, Red

    for node_idx in graph.nodes():
        x, y = pos_normalized[node_idx]
        linker = linkers[state[node_idx]]
        color = colors.get(linker, 'gray')
        circle = plt.Circle((x, y), 0.04, color=color, alpha=0.95, ec='black', linewidth=2, zorder=10)
        ax.add_patch(circle)
        ax.text(x, y, linker[0], ha='center', va='center', fontsize=10, fontweight='bold', color='white', zorder=11)

    ax.set_aspect('equal')
    ax.axis('off')

    title_text = f'{title}\nEnergy = {show_energy:.6f}' if show_energy is not None else title
    ax.set_title(title_text, fontsize=14, fontweight='bold', pad=20)

    legend_elements = [
        Patch(facecolor='#2ca02c', edgecolor='black', label='THQ (2.42Å)'),
        Patch(facecolor='#d62728', edgecolor='black', label='HHTP (4.87Å)'),
        Line2D([0], [0], color='#333333', linewidth=2.5, label='Topological'),
        Line2D([0], [0], color='#BBBBBB', linewidth=0.8, label='Spatial'),
    ]
    ax.legend(handles=legend_elements, loc='upper center',
             bbox_to_anchor=(0.5, -0.05), ncol=4, fontsize=9, frameon=True, fancybox=True, shadow=True)

    plt.tight_layout()
    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"  Saved: {filename}")
        plt.close(fig)

def run_pt_benchmark():
    np.random.seed(SEED)
    print("="*70)
    print(f"16-Site Benchmark (Parallel Tempering)")
    print("="*70)

    # 2x4 periodic honeycomb = 16 nodes
    G16_original = nx.hexagonal_lattice_graph(2, 4, periodic=True)
    G16 = G16_original.copy()
    G16 = add_edge_weights_to_graph(G16, MATERIAL_PARAMS)

    solver = ParallelTemperingSolver(G16, COUNTS_16, LENGTHS_16, original_graph=G16_original)

    print("\n" + "="*70)
    print(f"OPTIMIZATION ({RUNS} independent PT runs)")
    print("="*70)

    energies = []
    best_states = []
    times = []

    for i in range(RUNS):
        print(f"\n[Run {i+1}/{RUNS}]")
        t0 = time.time()
        e, state = solver.solve(steps=TOTAL_MC_STEPS, return_state=True)
        t1 = time.time()

        times.append(t1 - t0)
        energies.append(e)
        best_states.append(state.copy())

        # Analyze alternation
        alt_count, total = solver.analyze_state(state)
        print(f"  Time: {t1-t0:.2f}s | Energy: {e:.6f} | Alternation: {alt_count}/{total} ({100*alt_count/total:.1f}%)")

    min_e = min(energies)
    success_threshold = min_e * 1.001  # Within 0.1% of best
    success_count = sum(1 for e in energies if e <= success_threshold)
    rate = success_count / RUNS
    avg_time = np.mean(times)

    print("\n" + "="*70)
    print("RESULTS (Parallel Tempering)")
    print("="*70)
    print(f"Min Energy Found:   {min_e:.6f}")
    print(f"Max Energy Found:   {max(energies):.6f}")
    print(f"Mean Energy:        {np.mean(energies):.6f}")
    print(f"Std Dev:            {np.std(energies):.6f}")
    print(f"Success Rate:       {rate*100:.1f}% ({success_count}/{RUNS})")
    print(f"Avg Time per Run:   {avg_time:.2f}s")

    if rate > 0:
        if rate >= 0.99:
            runs_needed = 1
        else:
            runs_needed = math.ceil(math.log(0.01) / math.log(1 - rate))
        print(f"Runs for 99% Conf:  {runs_needed}")
        print(f"Total Time-to-Solution: {runs_needed * avg_time:.2f}s")

    best_idx = energies.index(min_e)
    best_state_overall = best_states[best_idx]
    alt_count, total = solver.analyze_state(best_state_overall)

    print(f"\n[Best Configuration Analysis]")
    print(f"  Alternation: {alt_count}/{total} topological edges ({100*alt_count/total:.1f}%)")
    if alt_count == total:
        print(f"  ✓ Perfect alternation achieved!")

    print("\n" + "="*70)
    print("GENERATING VISUALIZATION")
    print("="*70)
    visualize_configuration(best_state_overall, solver.graph, TYPES_16,
                          title="Lowest Energy (Parallel Tempering)",
                          filename="pt_ground_state_16site.png",
                          show_energy=min_e)
    print("="*70)

if __name__ == "__main__":
    run_pt_benchmark()

16-Site Benchmark (Parallel Tempering)

[Bipartite Structure Detected]
  Sublattice A: 8 nodes
  Sublattice B: 8 nodes

[Parallel Tempering Setup]
  Replicas: 16
  T_min: 0.10, T_max: 100.00
  Beta range: [10.0000, 0.0100]

OPTIMIZATION (10 independent PT runs)

[Run 1/10]
  Exchange rate: 84.2% (1257/1492)
  Time: 9.64s | Energy: 85.431934 | Alternation: 24/24 (100.0%)

[Run 2/10]
  Exchange rate: 84.0% (1254/1492)
  Time: 7.75s | Energy: 85.431934 | Alternation: 24/24 (100.0%)

[Run 3/10]
  Exchange rate: 90.3% (1347/1492)
  Time: 8.96s | Energy: 85.431934 | Alternation: 24/24 (100.0%)

[Run 4/10]
  Exchange rate: 84.8% (1265/1492)
  Time: 8.89s | Energy: 85.431934 | Alternation: 24/24 (100.0%)

[Run 5/10]
  Exchange rate: 90.5% (1351/1492)
  Time: 7.83s | Energy: 85.431934 | Alternation: 24/24 (100.0%)

[Run 6/10]
  Exchange rate: 83.2% (1242/1492)
  Time: 8.76s | Energy: 85.431934 | Alternation: 24/24 (100.0%)

[Run 7/10]
  Exchange rate: 85.7% (1279/1492)
  Time: 8.75s | Energy: 8

In [ ]:
# 32-Site Benchmark - Parallel Tempering (Replica Exchange)

import networkx as nx
import numpy as np
import time
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# --- CONFIGURATION ---
SEED = 42
RUNS = 10
TOTAL_MC_STEPS = 20000

# Parallel Tempering Parameters
NUM_REPLICAS = 16
T_MIN = 0.1
T_MAX = 100.0
EXCHANGE_INTERVAL = 100

# Material Parameters
MATERIAL_PARAMS = {
    'topological': {'alpha': 1.0, 'distance': 3.0},
    'spatial': {'alpha': 0.01, 'distance': 5.2}
}

# 32-Site: 8 Linker Types (4 of each)
TYPES_32 = ['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8']
COUNTS_32 = {t: 4 for t in TYPES_32}
LENGTHS_32 = {
    'L1': 2.0, 'L2': 3.0, 'L3': 4.0, 'L4': 5.0,
    'L5': 6.0, 'L6': 7.0, 'L7': 8.0, 'L8': 9.0
}

def add_edge_weights_to_graph(G, params):
    """Distance-based spatial edge detection"""
    t_p = params['topological']
    s_p = params['spatial']
    w_topo = t_p['distance'] ** t_p['alpha']
    w_spatial = s_p['distance'] ** s_p['alpha']

    pos = nx.get_node_attributes(G, 'pos')
    if not pos:
        pos = {node: data['pos'] for node, data in G.nodes(data=True) if 'pos' in data}

    topo_distances = []
    for u, v in G.edges():
        G[u][v]['weight'] = w_topo
        G[u][v]['edge_type'] = 'topological'
        pos_u = np.array(pos[u])
        pos_v = np.array(pos[v])
        dist = np.linalg.norm(pos_u - pos_v)
        topo_distances.append(dist)

    avg_topo_dist = np.mean(topo_distances)
    spatial_distance_min = 1.4 * avg_topo_dist
    spatial_distance_max = 2.1 * avg_topo_dist

    nodes = list(G.nodes())
    spatial_edges = set()

    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            if G.has_edge(u, v):
                continue
            pos_u = np.array(pos[u])
            pos_v = np.array(pos[v])
            dist = np.linalg.norm(pos_u - pos_v)
            if spatial_distance_min <= dist <= spatial_distance_max:
                spatial_edges.add((u, v))

    for u, v in spatial_edges:
        G.add_edge(u, v, weight=w_spatial, edge_type='spatial')

    return G

class ParallelTemperingSolver:
    def __init__(self, graph, target_counts, linker_lengths):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        self.graph = nx.relabel_nodes(graph, mapping)

        self.edges = np.array(list(self.graph.edges()), dtype=int)
        self.linkers = list(target_counts.keys())
        self.lengths = linker_lengths
        self.length_lookup = np.array([linker_lengths[l] for l in self.linkers])
        self.edge_weights = np.array([self.graph[i][j].get('weight', 1.0) for i, j in self.edges])

        self.state_pool = []
        for linker_idx, linker_name in enumerate(self.linkers):
            self.state_pool.extend([linker_idx] * target_counts[linker_name])
        self.state_pool = np.array(self.state_pool)
        self.n_sites = len(self.state_pool)

        # Temperature ladder (geometric spacing)
        self.temperatures = np.logspace(np.log10(T_MIN), np.log10(T_MAX), NUM_REPLICAS)
        self.betas = 1.0 / self.temperatures

        print(f"\n[Parallel Tempering Setup]")
        print(f"  Replicas: {NUM_REPLICAS}")
        print(f"  T_min: {T_MIN:.2f}, T_max: {T_MAX:.2f}")
        print(f"  Beta range: [{self.betas[0]:.4f}, {self.betas[-1]:.4f}]")

    def get_energy(self, current_state):
        u = self.edges[:, 0]
        v = self.edges[:, 1]
        len_u = self.length_lookup[current_state[u]]
        len_v = self.length_lookup[current_state[v]]
        edge_lengths = len_u + len_v
        mean_length = np.mean(edge_lengths)
        return np.sum(self.edge_weights * (edge_lengths - mean_length) ** 2)

    def solve(self, steps=TOTAL_MC_STEPS, return_state=False):
        # Initialize replicas - each gets independent random configuration
        replicas = []
        energies = []

        for _ in range(NUM_REPLICAS):
            state = self.state_pool.copy()
            np.random.shuffle(state)
            replicas.append(state.copy())
            energies.append(self.get_energy(state))

        # Track best solution across all replicas
        best_energy_global = min(energies)
        best_state_global = replicas[energies.index(best_energy_global)].copy()

        # Statistics tracking
        exchange_attempts = 0
        exchange_accepts = 0

        # Main loop
        for step in range(steps):
            # 1. Monte Carlo moves for each replica
            for i in range(NUM_REPLICAS):
                # Propose swap move
                idx1, idx2 = np.random.randint(0, self.n_sites, 2)
                if replicas[i][idx1] == replicas[i][idx2]:
                    continue

                current_E = energies[i]

                # Perform swap
                replicas[i][idx1], replicas[i][idx2] = replicas[i][idx2], replicas[i][idx1]
                new_E = self.get_energy(replicas[i])
                delta_E = new_E - current_E

                # Metropolis acceptance at temperature T_i
                beta = self.betas[i]
                if delta_E < 0 or np.random.rand() < np.exp(-beta * delta_E):
                    # Accept
                    energies[i] = new_E
                    if new_E < best_energy_global:
                        best_energy_global = new_E
                        best_state_global = replicas[i].copy()
                else:
                    # Reject - revert swap
                    replicas[i][idx1], replicas[i][idx2] = replicas[i][idx2], replicas[i][idx1]

            # 2. Replica exchange attempts
            if step % EXCHANGE_INTERVAL == 0 and step > 0:
                # Proper exchange protocol
                # Alternate between even (0-1, 2-3, ...) and odd (1-2, 3-4, ...) pairs
                offset = (step // EXCHANGE_INTERVAL) % 2

                for i in range(offset, NUM_REPLICAS - 1, 2):
                    j = i + 1

                    beta_i = self.betas[i]
                    beta_j = self.betas[j]
                    E_i = energies[i]
                    E_j = energies[j]

                    # Since i < j, we have T_i < T_j (lower index = lower temp)
                    # So β_i > β_j
                    delta_beta = beta_i - beta_j  # This is positive
                    delta_E = E_j - E_i

                    # Exchange probability
                    log_acceptance = delta_beta * delta_E

                    exchange_attempts += 1

                    if log_acceptance >= 0 or np.random.rand() < np.exp(log_acceptance):
                        # Accept exchange - swap states
                        replicas[i], replicas[j] = replicas[j], replicas[i]
                        energies[i], energies[j] = energies[j], energies[i]
                        exchange_accepts += 1

        # Print exchange statistics
        if exchange_attempts > 0:
            exchange_rate = 100.0 * exchange_accepts / exchange_attempts
            print(f"  Exchange rate: {exchange_rate:.1f}% ({exchange_accepts}/{exchange_attempts})")

        if return_state:
            return best_energy_global, best_state_global
        return best_energy_global

def visualize_configuration(state, graph, linkers, lengths, title="Configuration", filename=None, show_energy=None):
    fig, ax = plt.subplots(figsize=(12, 10))

    if not all(isinstance(node, int) for node in graph.nodes()):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        graph = nx.relabel_nodes(graph, mapping)

    pos = nx.get_node_attributes(graph, 'pos')
    if not pos:
        pos = nx.spring_layout(graph, seed=SEED)

    pos_array = np.array(list(pos.values()))
    pos_min = pos_array.min(axis=0)
    pos_max = pos_array.max(axis=0)
    pos_range = pos_max - pos_min
    pos_normalized = {node: (p - pos_min) / pos_range for node, p in pos.items()}

    for i, j, data in graph.edges(data=True):
        edge_type = data.get('edge_type', 'unknown')
        x_coords = [pos_normalized[i][0], pos_normalized[j][0]]
        y_coords = [pos_normalized[i][1], pos_normalized[j][1]]

        if edge_type == 'topological':
            ax.plot(x_coords, y_coords, color='#333333', linewidth=2.0, alpha=0.8, zorder=5)
        else:
            ax.plot(x_coords, y_coords, color='#CCCCCC', linewidth=0.5, alpha=0.3, zorder=1)

    # Use colormap for 8 linker types
    colors_map = plt.cm.tab10(np.linspace(0, 1, len(set(linkers))))
    colors = {linker: colors_map[i] for i, linker in enumerate(sorted(set(linkers)))}

    for node_idx in graph.nodes():
        x, y = pos_normalized[node_idx]
        linker = linkers[state[node_idx]]
        color = colors.get(linker, 'gray')
        circle = plt.Circle((x, y), 0.03, color=color, alpha=0.95, ec='black', linewidth=1.5, zorder=10)
        ax.add_patch(circle)
        ax.text(x, y, linker, ha='center', va='center', fontsize=6, fontweight='bold', color='white', zorder=11)

    ax.set_aspect('equal')
    ax.axis('off')

    title_text = f'{title}\nEnergy = {show_energy:.6f}' if show_energy is not None else title
    ax.set_title(title_text, fontsize=14, fontweight='bold', pad=20)

    legend_elements = [Patch(facecolor=colors[l], edgecolor='black',
                            label=f'{l} ({lengths[l]:.1f}Å)') for l in sorted(set(linkers))]
    ax.legend(handles=legend_elements, loc='upper center',
             bbox_to_anchor=(0.5, -0.02), ncol=4, fontsize=8, frameon=True)

    plt.tight_layout()
    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"  Saved: {filename}")
        plt.close(fig)

def run_pt_benchmark():
    np.random.seed(SEED)
    print("="*70)
    print(f"32-Site Benchmark (Parallel Tempering)")
    print("="*70)

    # 4x4 periodic honeycomb = 32 nodes
    G32 = nx.hexagonal_lattice_graph(4, 4, periodic=True)
    G32 = add_edge_weights_to_graph(G32, MATERIAL_PARAMS)

    solver = ParallelTemperingSolver(G32, COUNTS_32, LENGTHS_32)

    print("\n" + "="*70)
    print(f"OPTIMIZATION ({RUNS} independent PT runs)")
    print("="*70)

    energies = []
    best_states = []
    times = []

    for i in range(RUNS):
        print(f"\n[Run {i+1}/{RUNS}]")
        t0 = time.time()
        e, state = solver.solve(steps=TOTAL_MC_STEPS, return_state=True)
        t1 = time.time()

        times.append(t1 - t0)
        energies.append(e)
        best_states.append(state.copy())
        print(f"  Time: {t1-t0:.2f}s | Energy: {e:.6f}")

    min_e = min(energies)
    success_threshold = min_e * 1.001  # Within 0.1% of best
    success_count = sum(1 for e in energies if e <= success_threshold)
    rate = success_count / RUNS
    avg_time = np.mean(times)

    print("\n" + "="*70)
    print("RESULTS (Parallel Tempering)")
    print("="*70)
    print(f"Min Energy Found:   {min_e:.6f}")
    print(f"Max Energy Found:   {max(energies):.6f}")
    print(f"Mean Energy:        {np.mean(energies):.6f}")
    print(f"Std Dev:            {np.std(energies):.6f}")
    print(f"Success Rate:       {rate*100:.1f}% ({success_count}/{RUNS})")
    print(f"Avg Time per Run:   {avg_time:.2f}s")

    if rate > 0:
        if rate >= 0.99:
            runs_needed = 1
        else:
            runs_needed = math.ceil(math.log(0.01) / math.log(1 - rate))
        print(f"Runs for 99% Conf:  {runs_needed}")
        print(f"Total Time-to-Solution: {runs_needed * avg_time:.2f}s")

    best_idx = energies.index(min_e)
    best_state_overall = best_states[best_idx]

    print("\n" + "="*70)
    print("GENERATING VISUALIZATION")
    print("="*70)
    visualize_configuration(best_state_overall, solver.graph, TYPES_32, LENGTHS_32,
                          title="Lowest Energy (Parallel Tempering)",
                          filename="pt_ground_state_32site.png",
                          show_energy=min_e)
    print("="*70)

if __name__ == "__main__":
    run_pt_benchmark()

32-Site Benchmark (Parallel Tempering)

[Parallel Tempering Setup]
  Replicas: 16
  T_min: 0.10, T_max: 100.00
  Beta range: [10.0000, 0.0100]

OPTIMIZATION (10 independent PT runs)

[Run 1/10]
  Exchange rate: 72.6% (1083/1492)
  Time: 16.49s | Energy: 1616.868367

[Run 2/10]
  Exchange rate: 73.5% (1096/1492)
  Time: 13.54s | Energy: 1616.868367

[Run 3/10]
  Exchange rate: 67.0% (999/1492)
  Time: 13.42s | Energy: 1623.067846

[Run 4/10]
  Exchange rate: 71.4% (1065/1492)
  Time: 13.42s | Energy: 1616.868367

[Run 5/10]
  Exchange rate: 77.3% (1153/1492)
  Time: 13.58s | Energy: 1616.868367

[Run 6/10]
  Exchange rate: 73.9% (1103/1492)
  Time: 13.91s | Energy: 1616.868367

[Run 7/10]
  Exchange rate: 74.7% (1114/1492)
  Time: 14.25s | Energy: 1616.868367

[Run 8/10]
  Exchange rate: 77.1% (1150/1492)
  Time: 13.57s | Energy: 1616.868367

[Run 9/10]
  Exchange rate: 74.1% (1106/1492)
  Time: 13.78s | Energy: 1627.034599

[Run 10/10]
  Exchange rate: 74.2% (1107/1492)
  Time: 14.05s 

In [ ]:
# 72-Site Benchmark - Parallel Tempering (Replica Exchange)

import networkx as nx
import numpy as np
import time
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# --- CONFIGURATION ---
SEED = 42
RUNS = 10
TOTAL_MC_STEPS = 20000

# Parallel Tempering Parameters
NUM_REPLICAS = 16
T_MIN = 0.1
T_MAX = 100.0
EXCHANGE_INTERVAL = 100

# Material Parameters
MATERIAL_PARAMS = {
    'topological': {'alpha': 1.0, 'distance': 3.0},
    'spatial': {'alpha': 0.01, 'distance': 5.2}
}

TYPES_72 = ['THQ', 'HHTP', 'HHTT', 'HHTN']
COUNTS_72 = {t: 18 for t in TYPES_72}
LENGTHS_72 = {'THQ': 2.0, 'HHTP': 4.0, 'HHTT': 6.0, 'HHTN': 8.0}

def add_edge_weights_to_graph(G, params):
    """Distance-based spatial edge detection"""
    t_p = params['topological']
    s_p = params['spatial']
    w_topo = t_p['distance'] ** t_p['alpha']
    w_spatial = s_p['distance'] ** s_p['alpha']

    pos = nx.get_node_attributes(G, 'pos')
    if not pos:
        pos = {node: data['pos'] for node, data in G.nodes(data=True) if 'pos' in data}

    topo_distances = []
    for u, v in G.edges():
        G[u][v]['weight'] = w_topo
        G[u][v]['edge_type'] = 'topological'
        pos_u = np.array(pos[u])
        pos_v = np.array(pos[v])
        dist = np.linalg.norm(pos_u - pos_v)
        topo_distances.append(dist)

    avg_topo_dist = np.mean(topo_distances)
    spatial_distance_min = 1.4 * avg_topo_dist
    spatial_distance_max = 2.1 * avg_topo_dist

    nodes = list(G.nodes())
    spatial_edges = set()

    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            if G.has_edge(u, v):
                continue
            pos_u = np.array(pos[u])
            pos_v = np.array(pos[v])
            dist = np.linalg.norm(pos_u - pos_v)
            if spatial_distance_min <= dist <= spatial_distance_max:
                spatial_edges.add((u, v))

    for u, v in spatial_edges:
        G.add_edge(u, v, weight=w_spatial, edge_type='spatial')

    return G

class ParallelTemperingSolver:
    def __init__(self, graph, target_counts, linker_lengths):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        self.graph = nx.relabel_nodes(graph, mapping)

        self.edges = np.array(list(self.graph.edges()), dtype=int)
        self.linkers = list(target_counts.keys())
        self.lengths = linker_lengths
        self.length_lookup = np.array([linker_lengths[l] for l in self.linkers])
        self.edge_weights = np.array([self.graph[i][j].get('weight', 1.0) for i, j in self.edges])

        self.state_pool = []
        for linker_idx, linker_name in enumerate(self.linkers):
            self.state_pool.extend([linker_idx] * target_counts[linker_name])
        self.state_pool = np.array(self.state_pool)
        self.n_sites = len(self.state_pool)

        # Temperature ladder (geometric spacing)
        self.temperatures = np.logspace(np.log10(T_MIN), np.log10(T_MAX), NUM_REPLICAS)
        self.betas = 1.0 / self.temperatures

        print(f"\n[Parallel Tempering Setup]")
        print(f"  Replicas: {NUM_REPLICAS}")
        print(f"  T_min: {T_MIN:.2f}, T_max: {T_MAX:.2f}")
        print(f"  Beta range: [{self.betas[0]:.4f}, {self.betas[-1]:.4f}]")

    def get_energy(self, current_state):
        u = self.edges[:, 0]
        v = self.edges[:, 1]
        len_u = self.length_lookup[current_state[u]]
        len_v = self.length_lookup[current_state[v]]
        edge_lengths = len_u + len_v
        mean_length = np.mean(edge_lengths)
        return np.sum(self.edge_weights * (edge_lengths - mean_length) ** 2)

    def solve(self, steps=TOTAL_MC_STEPS, return_state=False):
        # Initialize replicas - each gets independent random configuration
        replicas = []
        energies = []

        for _ in range(NUM_REPLICAS):
            state = self.state_pool.copy()
            np.random.shuffle(state)
            replicas.append(state.copy())  # FIX: Use .copy()
            energies.append(self.get_energy(state))

        # Track best solution across all replicas
        best_energy_global = min(energies)
        best_state_global = replicas[energies.index(best_energy_global)].copy()

        # Statistics tracking
        exchange_attempts = 0
        exchange_accepts = 0

        # Main loop
        for step in range(steps):
            # 1. Monte Carlo moves for each replica
            for i in range(NUM_REPLICAS):
                # Propose swap move
                idx1, idx2 = np.random.randint(0, self.n_sites, 2)
                if replicas[i][idx1] == replicas[i][idx2]:
                    continue

                current_E = energies[i]

                # Perform swap
                replicas[i][idx1], replicas[i][idx2] = replicas[i][idx2], replicas[i][idx1]
                new_E = self.get_energy(replicas[i])
                delta_E = new_E - current_E

                # Metropolis acceptance at temperature T_i
                beta = self.betas[i]
                if delta_E < 0 or np.random.rand() < np.exp(-beta * delta_E):
                    # Accept
                    energies[i] = new_E
                    if new_E < best_energy_global:
                        best_energy_global = new_E
                        best_state_global = replicas[i].copy()
                else:
                    # Reject - revert swap
                    replicas[i][idx1], replicas[i][idx2] = replicas[i][idx2], replicas[i][idx1]

            # 2. Replica exchange attempts
            if step % EXCHANGE_INTERVAL == 0 and step > 0:
                # CRITICAL FIX: Proper exchange protocol
                # Try swapping adjacent temperature pairs
                # Alternate between even (0-1, 2-3, ...) and odd (1-2, 3-4, ...) pairs
                offset = (step // EXCHANGE_INTERVAL) % 2

                for i in range(offset, NUM_REPLICAS - 1, 2):
                    j = i + 1

                    # CORRECT EXCHANGE PROBABILITY:
                    # For swapping configurations between replicas at T_i and T_j:
                    # P_exchange = min(1, exp[(β_i - β_j)(E_j - E_i)])
                    # where β = 1/T

                    beta_i = self.betas[i]
                    beta_j = self.betas[j]
                    E_i = energies[i]
                    E_j = energies[j]

                    # Since i < j, we have T_i < T_j (lower index = lower temp)
                    # So β_i > β_j
                    delta_beta = beta_i - beta_j  # This is positive
                    delta_E = E_j - E_i

                    # Exchange probability
                    log_acceptance = delta_beta * delta_E

                    exchange_attempts += 1

                    if log_acceptance >= 0 or np.random.rand() < np.exp(log_acceptance):
                        # Accept exchange - swap STATES (not energies!)
                        # FIX: Must swap the actual state arrays
                        replicas[i], replicas[j] = replicas[j], replicas[i]
                        energies[i], energies[j] = energies[j], energies[i]
                        exchange_accepts += 1

        # Print exchange statistics
        if exchange_attempts > 0:
            exchange_rate = 100.0 * exchange_accepts / exchange_attempts
            print(f"  Exchange rate: {exchange_rate:.1f}% ({exchange_accepts}/{exchange_attempts})")

        if return_state:
            return best_energy_global, best_state_global
        return best_energy_global

def visualize_configuration(state, graph, linkers, title="Configuration", filename=None, show_energy=None):
    fig, ax = plt.subplots(figsize=(12, 10))

    if not all(isinstance(node, int) for node in graph.nodes()):
        sorted_nodes = sorted(list(graph.nodes()))
        mapping = {node: i for i, node in enumerate(sorted_nodes)}
        graph = nx.relabel_nodes(graph, mapping)

    pos = nx.get_node_attributes(graph, 'pos')
    if not pos:
        pos = nx.spring_layout(graph, seed=SEED)

    pos_array = np.array(list(pos.values()))
    pos_min = pos_array.min(axis=0)
    pos_max = pos_array.max(axis=0)
    pos_range = pos_max - pos_min
    pos_normalized = {node: (p - pos_min) / pos_range for node, p in pos.items()}

    for i, j, data in graph.edges(data=True):
        edge_type = data.get('edge_type', 'unknown')
        x_coords = [pos_normalized[i][0], pos_normalized[j][0]]
        y_coords = [pos_normalized[i][1], pos_normalized[j][1]]

        if edge_type == 'topological':
            ax.plot(x_coords, y_coords, color='#333333', linewidth=1.5, alpha=0.7, zorder=5)
        else:
            ax.plot(x_coords, y_coords, color='#DDDDDD', linewidth=0.5, alpha=0.3, zorder=1)

    colors = {
        'THQ': '#2ca02c', 'HHTP': '#d62728',
        'HHTT': '#1f77b4', 'HHTN': '#ff7f0e'
    }

    for node_idx in graph.nodes():
        x, y = pos_normalized[node_idx]
        linker = linkers[state[node_idx]]
        color = colors.get(linker, 'gray')
        circle = plt.Circle((x, y), 0.02, color=color, alpha=0.95, ec='black', linewidth=1, zorder=10)
        ax.add_patch(circle)

    ax.set_aspect('equal')
    ax.axis('off')

    title_text = f'{title}\nEnergy = {show_energy:.6f}' if show_energy is not None else title
    ax.set_title(title_text, fontsize=14, fontweight='bold', pad=20)

    legend_elements = [Patch(facecolor=colors[l], edgecolor='black',
                            label=f'{l} ({LENGTHS_72[l]:.1f}Å)') for l in TYPES_72]
    ax.legend(handles=legend_elements, loc='upper center',
             bbox_to_anchor=(0.5, -0.02), ncol=4, fontsize=9, frameon=True)

    plt.tight_layout()
    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"  Saved: {filename}")
        plt.close(fig)

def run_pt_benchmark():
    np.random.seed(SEED)
    print("="*70)
    print(f"72-Site Benchmark (Parallel Tempering)")
    print("="*70)

    G72 = nx.hexagonal_lattice_graph(6, 6, periodic=True)
    G72 = add_edge_weights_to_graph(G72, MATERIAL_PARAMS)

    solver = ParallelTemperingSolver(G72, COUNTS_72, LENGTHS_72)

    print("\n" + "="*70)
    print(f"OPTIMIZATION ({RUNS} independent PT runs)")
    print("="*70)

    energies = []
    best_states = []
    times = []

    for i in range(RUNS):
        print(f"\n[Run {i+1}/{RUNS}]")
        t0 = time.time()
        e, state = solver.solve(steps=TOTAL_MC_STEPS, return_state=True)
        t1 = time.time()

        times.append(t1 - t0)
        energies.append(e)
        best_states.append(state.copy())
        print(f"  Time: {t1-t0:.2f}s | Energy: {e:.6f}")

    min_e = min(energies)
    success_threshold = min_e * 1.001  # Within 0.1% of best
    success_count = sum(1 for e in energies if e <= success_threshold)
    rate = success_count / RUNS
    avg_time = np.mean(times)

    print("\n" + "="*70)
    print("RESULTS (Parallel Tempering)")
    print("="*70)
    print(f"Min Energy Found:   {min_e:.6f}")
    print(f"Max Energy Found:   {max(energies):.6f}")
    print(f"Mean Energy:        {np.mean(energies):.6f}")
    print(f"Std Dev:            {np.std(energies):.6f}")
    print(f"Success Rate:       {rate*100:.1f}% ({success_count}/{RUNS})")
    print(f"Avg Time per Run:   {avg_time:.2f}s")

    if rate > 0:
        if rate >= 0.99:
            runs_needed = 1
        else:
            runs_needed = math.ceil(math.log(0.01) / math.log(1 - rate))
        print(f"Runs for 99% Conf:  {runs_needed}")
        print(f"Total Time-to-Solution: {runs_needed * avg_time:.2f}s")

    best_idx = energies.index(min_e)
    best_state_overall = best_states[best_idx]

    print("\n" + "="*70)
    print("GENERATING VISUALIZATION")
    print("="*70)
    visualize_configuration(best_state_overall, solver.graph, TYPES_72,
                          title="Lowest Energy (Parallel Tempering)",
                          filename="pt_ground_state_72site.png",
                          show_energy=min_e)
    print("="*70)

if __name__ == "__main__":
    run_pt_benchmark()

72-Site Benchmark (Parallel Tempering)

[Parallel Tempering Setup]
  Replicas: 16
  T_min: 0.10, T_max: 100.00
  Beta range: [10.0000, 0.0100]

OPTIMIZATION (10 independent PT runs)

[Run 1/10]
  Exchange rate: 84.8% (1265/1492)
  Time: 14.70s | Energy: 4446.422140

[Run 2/10]
  Exchange rate: 86.8% (1295/1492)
  Time: 15.27s | Energy: 4462.289155

[Run 3/10]
  Exchange rate: 88.2% (1316/1492)
  Time: 15.00s | Energy: 4486.254543

[Run 4/10]
  Exchange rate: 84.5% (1260/1492)
  Time: 14.77s | Energy: 4443.924558

[Run 5/10]
  Exchange rate: 85.7% (1279/1492)
  Time: 14.62s | Energy: 4474.822694

[Run 6/10]
  Exchange rate: 88.2% (1316/1492)
  Time: 15.28s | Energy: 4458.869150

[Run 7/10]
  Exchange rate: 86.5% (1290/1492)
  Time: 14.59s | Energy: 4434.209682

[Run 8/10]
  Exchange rate: 87.0% (1298/1492)
  Time: 15.02s | Energy: 4485.961261

[Run 9/10]
  Exchange rate: 88.6% (1322/1492)
  Time: 15.24s | Energy: 4438.688112

[Run 10/10]
  Exchange rate: 85.7% (1278/1492)
  Time: 14.62s